LIFECYCLE PORTFOLIO CHOICE MODEL - PART 1
Model Definition and Precomputation

In [15]:
"""
LIFECYCLE PORTFOLIO CHOICE MODEL - PART 1
Model Definition and Precomputation

Generic architecture: full VAR variables are partitioned into
- state variables (discretized on DP grid)
- return variables (integrated conditionally on state transitions)

This cell defines:
- LifecyclePortfolioModel
- VAR partitioning utilities
- Income/tax/discretization helpers
- Precompute class with generic state-return logic
- build_model() factory for config-driven initialization
"""

import numpy as np
from scipy.special import roots_hermite
from scipy.stats import norm
from typing import NamedTuple, Callable
from earnings_dependent_mortality import calibrate_earnings_dependent_mortality


# =============================================================================
# 1. MODEL CLASS                                                               
# =============================================================================

class LifecyclePortfolioModel(NamedTuple):
    """Model specification with generic state-return partition."""

    # Preferences
    u: Callable
    u_prime: Callable
    u_prime_inv: Callable
    beta: float
    gamma: float

    # Bequest
    b_bar: int       # bequest horizon in years (Catherine 2025: 10)

    # Lifecycle
    start_age: int
    retire_age: int
    terminal_age: int
    # survival_probs: moved to Precompute as survival_probs_2d (n_age, n_z)

    # Labor income (Catherine 2025 / Guvenen et al. 2022)
    b0: float            # age-earnings intercept
    b1: float            # age-earnings linear
    b2: float            # age-earnings quadratic (/10)
    b3: float            # age-earnings cubic (/100)
    rho: float
    pz: float
    mu_eta1: float
    sigma_eta1: float
    mu_eta2: float
    sigma_eta2: float
    pe: float
    mu_eps1: float
    sigma_eps1: float
    mu_eps2: float   # NOTE: overridden in get_eps_quadrature_corrected (zero-mean enforcement)
    sigma_eps2: float

    # Partitioned VAR structure
    n_state: int
    n_ret: int
    state_names: tuple
    ret_names: tuple

    z_bar_state: np.ndarray
    z_bar_ret: np.ndarray

    Phi_0_state: np.ndarray
    Phi_11: np.ndarray
    Phi_0_ret: np.ndarray
    Phi_21: np.ndarray

    Sigma_ss: np.ndarray
    Sigma_rr: np.ndarray
    Sigma_rs: np.ndarray
    M: np.ndarray
    Sigma_r_cond: np.ndarray

    bill_rate_index_in_state: int
    annuity_yield_index_in_state: int

    # Portfolio constraints
    constrained: bool            # True = no short-selling/leverage, False = unconstrained


# =============================================================================
# DISCRETIZATION CONFIG
# =============================================================================

class DiscretizationConfig(NamedTuple):
    """Discretization choices for grids and quadrature. Passed to Precompute."""

    # Wealth grid
    n_wealth: int = 150
    wealth_min: float = 1e-4
    wealth_max: float = 200.0

    # Savings grid (EGM)
    n_savings: int = 150
    savings_min: float = 1e-8

    # Financial state VAR discretization
    state_grid_sizes: tuple = (5, 5, 5)

    # Income process
    n_z: int = 11                       # persistent income grid points
    n_eps_nodes: int = 5                # Gauss-Hermite nodes (doubled internally)

    # Validation tolerances for Rouwenhorst consistency check
    consistency_tol_warn: float = 2e-2
    consistency_tol_error: float = 1e-1


# =============================================================================
# SOLVER CONFIG
# =============================================================================

class SolverConfig(NamedTuple):
    """Newton solver numerical choices. Passed to run_lifecycle_solver."""

    # --- Newton iteration ---
    tol: float = 1e-7                         # FOC convergence tolerance
    max_iter: int = 20                         # max Newton iterations (constrained)
    max_iter_unconstrained: int = 30           # max Newton iterations (unconstrained)
    edge_max_iter: int = 8                     # max iterations for 1D edge Newton

    # --- Initial guess ---
    init_alpha_s: float = 0.1                  # initial stock weight guess
    init_alpha_b: float = 0.4                  # initial bond weight guess

    # --- Step control ---
    step_damp_constrained: float = 0.2         # max Newton step length (constrained)
    step_damp_unconstrained: float = 0.3       # max Newton step length (unconstrained)
    grad_step_size: float = 0.05               # gradient descent step when Jacobian singular

    # --- Thresholds ---
    tiny_savings: float = 1e-6                 # below this, skip solver (all-bills)
    corner_tol: float = 1e-8                   # KKT tolerance multiplier for corner acceptance
    edge_accept_factor: float = 10.0           # edge acceptance = tol * scale * this factor
    singular_det: float = 1e-15                # Jacobian determinant singularity threshold
    grad_denom_eps: float = 1e-10              # epsilon in gradient fallback denominator

    # --- Safety clamps ---
    min_wealth_inv: float = 1e-10              # floor for max(s_val * R_p, ...)
    min_consumption: float = 1e-10             # floor for max(c_next, ...)
    min_return_power: float = 1e-15            # floor for max(R_p, ...) before power

    # --- Probability / EGM / Euler ---
    prob_skip_threshold: float = 1e-12         # skip states with prob below this
    euler_inv_floor: float = 1e-20             # floor for beta*euler before inversion
    egm_anchor: float = 1e-10                  # anchor value for EGM grid at zero savings


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def mixture_cdf(x, p, mu1, sigma1, mu2, sigma2):
    """CDF of two-component normal mixture."""
    return p * norm.cdf(x, loc=mu1, scale=sigma1) + (1.0 - p) * norm.cdf(x, loc=mu2, scale=sigma2)


def mixture_quantile(q, p, mu1, sigma1, mu2, sigma2, tol=1e-10, max_iter=100):
    """Quantile of normal mixture via bisection."""
    lower = min(mu1 - 6.0 * sigma1, mu2 - 6.0 * sigma2)
    upper = max(mu1 + 6.0 * sigma1, mu2 + 6.0 * sigma2)

    for _ in range(max_iter):
        mid = 0.5 * (lower + upper)
        cdf_val = mixture_cdf(mid, p, mu1, sigma1, mu2, sigma2)
        if abs(cdf_val - q) < tol:
            return mid
        if cdf_val < q:
            lower = mid
        else:
            upper = mid
    return 0.5 * (lower + upper)


def create_utility_functions(gamma):
    """Create CRRA utility functions for given risk aversion."""
    if gamma == 1.0:
        def u(c):
            return np.log(c)

        def u_prime(c):
            return 1.0 / c

        def u_prime_inv(mu):
            return 1.0 / mu
    else:
        def u(c):
            return c ** (1.0 - gamma) / (1.0 - gamma)

        def u_prime(c):
            return c ** (-gamma)

        def u_prime_inv(mu):
            return mu ** (-1.0 / gamma)

    return u, u_prime, u_prime_inv


# =============================================================================
# BEQUEST UTILITY FUNCTIONS  (Catherine 2025, equations 21-22)
# =============================================================================

def annuity_factor(y_ann, b_bar):
    """
    Standard fixed-rate annuity factor: PV of b_bar annual payments of 1
    discounted at the annual nominal yield y_ann.

    A(y) = sum_{k=1}^{b_bar} (1+y)^{-k} = (1 - (1+y)^{-b_bar}) / y

    Using the 10-year nominal bond yield (y_nom * 4) as the discount rate is
    coherent because the bequest horizon b_bar equals the bond maturity: the
    heir receives a consumption stream of the same length and the nominal bond
    is the natural pricing instrument for that stream.

    Parameters
    ----------
    y_ann : float or array  Annual nominal yield (e.g. y_nom_quarterly * 4).
    b_bar : int             Bequest horizon in years (= bond maturity = 10).
    """
    y_ann = np.asarray(y_ann, dtype=float)
    return (1.0 - (1.0 + y_ann) ** (-b_bar)) / y_ann


def bequest_utility(W, A, gamma, b_bar):
    """
    Bequest utility:  b(W, r_f) = b_bar * (W / A)^(1 - gamma) / (1 - gamma)

    where  A = annuity_factor(r_f, ...)  is precomputed for the relevant state
    and    C_bar = W / A  is the flow-equivalent consumption implied by wealth W
    spread over b_bar annuity periods.

    Parameters
    ----------
    W     : float or array  End-of-period wealth (bequest).
    A     : float or array  Annuity factor A(r_f, b_bar) at current financial state.
    gamma : float           CRRA risk aversion.
    b_bar : int             Bequest weight / horizon (Catherine 2025: 10).
    """
    C_bar = W / A
    return b_bar * C_bar**(1.0 - gamma) / (1.0 - gamma)


def bequest_marginal(W, A, gamma, b_bar):
    """
    Marginal bequest utility:  db/dW = b_bar * (W / A)^(-gamma) / A

    Parameters
    ----------
    W     : float or array  End-of-period wealth.
    A     : float or array  Annuity factor at current financial state.
    gamma : float           CRRA risk aversion.
    b_bar : int             Bequest weight / horizon.
    """
    C_bar = W / A
    return b_bar * C_bar**(-gamma) / A


def bequest_marginal_inv(mu, A, gamma, b_bar):
    """
    Inverse of bequest_marginal: given  mu = db/dW,  solve for W.

    W = A * (mu * A / b_bar)^(-1/gamma)

    Used in the EGM terminal condition: the period-T+1 "value" is bequest
    utility, so the marginal value of wealth is bequest_marginal(W, A, ...).
    Inverting gives the optimal terminal wealth as a function of the shadow
    price mu.
    """
    return A * (mu * A / b_bar)**(-1.0 / gamma)


def disposable_income_working(y_gross):
    """After-tax labor income using the same progressive schedule as prior model."""
    y = np.asarray(y_gross, dtype=float)

    payroll_tax = 0.106 * np.minimum(y, 2.5)
    taxable_income = np.maximum(0.0, y - payroll_tax)

    tax = np.zeros_like(taxable_income)
    m = taxable_income <= 0.18
    tax[m] = taxable_income[m] * 0.10
    m = (taxable_income > 0.18) & (taxable_income <= 0.72)
    tax[m] = 0.018 + (taxable_income[m] - 0.18) * 0.12
    m = (taxable_income > 0.72) & (taxable_income <= 1.54)
    tax[m] = 0.0828 + (taxable_income[m] - 0.72) * 0.22
    m = (taxable_income > 1.54) & (taxable_income <= 2.94)
    tax[m] = 0.2632 + (taxable_income[m] - 1.54) * 0.24
    m = (taxable_income > 2.94) & (taxable_income <= 3.73)
    tax[m] = 0.5992 + (taxable_income[m] - 2.94) * 0.32
    m = (taxable_income > 3.73) & (taxable_income <= 9.32)
    tax[m] = 0.8520 + (taxable_income[m] - 3.73) * 0.35
    m = taxable_income > 9.32
    tax[m] = 2.8085 + (taxable_income[m] - 9.32) * 0.37

    return taxable_income - tax


def compute_pension_after_tax(z_grid):
    """Social Security benefits with progressive formula and taxes."""
    z = np.asarray(z_grid, dtype=float)
    career_rank = np.exp(z)
    pension = np.zeros_like(career_rank)

    m = career_rank <= 0.21
    pension[m] = career_rank[m] * 0.90
    m = (career_rank > 0.21) & (career_rank <= 1.25)
    pension[m] = 0.189 + (career_rank[m] - 0.21) * 0.32
    m = career_rank > 1.25
    pension[m] = 0.5218 + (career_rank[m] - 1.25) * 0.15

    tax = np.zeros_like(pension)
    m = pension <= 0.18
    tax[m] = pension[m] * 0.10
    m = (pension > 0.18) & (pension <= 0.72)
    tax[m] = 0.018 + (pension[m] - 0.18) * 0.12
    m = (pension > 0.72) & (pension <= 1.54)
    tax[m] = 0.0828 + (pension[m] - 0.72) * 0.22
    m = (pension > 1.54) & (pension <= 2.94)
    tax[m] = 0.2632 + (pension[m] - 1.54) * 0.24
    m = (pension > 2.94) & (pension <= 3.73)
    tax[m] = 0.5992 + (pension[m] - 2.94) * 0.32
    m = (pension > 3.73) & (pension <= 9.32)
    tax[m] = 0.8520 + (pension[m] - 3.73) * 0.35
    m = pension > 9.32
    tax[m] = 2.8085 + (pension[m] - 9.32) * 0.37

    return pension - tax


In [16]:
# =============================================================================
# 3. VAR PARTITION (GENERIC)
# =============================================================================

def _validate_partition_inputs(Phi_full, Omega_full, z_bar, state_idx, ret_idx):
    n = len(z_bar)
    if Phi_full.shape != (n, n):
        raise ValueError(f"Phi must have shape {(n, n)}, got {Phi_full.shape}")
    if Omega_full.shape != (n, n):
        raise ValueError(f"Omega must have shape {(n, n)}, got {Omega_full.shape}")

    state_idx = np.asarray(state_idx, dtype=int)
    ret_idx = np.asarray(ret_idx, dtype=int)

    if len(state_idx) == 0 or len(ret_idx) == 0:
        raise ValueError("state_idx and ret_idx must both be non-empty")

    all_idx = np.concatenate([state_idx, ret_idx])
    if np.any(all_idx < 0) or np.any(all_idx >= n):
        raise ValueError("state_idx/ret_idx contains out-of-bounds index")

    if len(np.unique(all_idx)) != len(all_idx):
        raise ValueError("state_idx and ret_idx overlap or contain duplicates")

    if len(np.unique(all_idx)) != n:
        missing = sorted(set(range(n)) - set(all_idx.tolist()))
        raise ValueError(f"state_idx + ret_idx must cover all variables exactly once. Missing: {missing}")

    return state_idx, ret_idx


def partition_var(Phi_full, Omega_full, z_bar, state_idx, ret_idx, variable_names=None, verbose=True):
    """Partition full VAR into state and return blocks using index lists."""
    Phi_full = np.asarray(Phi_full, dtype=float)
    Omega_full = np.asarray(Omega_full, dtype=float)
    z_bar = np.asarray(z_bar, dtype=float)

    state_idx, ret_idx = _validate_partition_inputs(Phi_full, Omega_full, z_bar, state_idx, ret_idx)

    if variable_names is None:
        variable_names = tuple(f"z{i}" for i in range(len(z_bar)))
    else:
        variable_names = tuple(variable_names)
        if len(variable_names) != len(z_bar):
            raise ValueError("variable_names length must match VAR dimension")

    Phi_11 = Phi_full[np.ix_(state_idx, state_idx)]
    Phi_21 = Phi_full[np.ix_(ret_idx, state_idx)]
    Phi_12 = Phi_full[np.ix_(state_idx, ret_idx)]
    Phi_22 = Phi_full[np.ix_(ret_idx, ret_idx)]

    Sigma_ss = Omega_full[np.ix_(state_idx, state_idx)]
    Sigma_rr = Omega_full[np.ix_(ret_idx, ret_idx)]
    Sigma_rs = Omega_full[np.ix_(ret_idx, state_idx)]
    Sigma_sr = Omega_full[np.ix_(state_idx, ret_idx)]

    M = Sigma_rs @ np.linalg.inv(Sigma_ss)
    Sigma_r_cond = Sigma_rr - M @ Sigma_sr

    z_bar_state = z_bar[state_idx]
    z_bar_ret = z_bar[ret_idx]

    # Intercepts: compute from the full VAR then partition.
    # This is exact for both restricted (Phi_12=0, Phi_22=0) and unrestricted VAR.
    Phi_0_full  = (np.eye(len(z_bar)) - Phi_full) @ z_bar
    Phi_0_state = Phi_0_full[state_idx]
    Phi_0_ret   = Phi_0_full[ret_idx]

    Phi_12_norm = np.linalg.norm(Phi_12)
    Phi_22_norm = np.linalg.norm(Phi_22)

    explained_share = 1.0 - np.clip(np.diag(Sigma_r_cond) / np.maximum(np.diag(Sigma_rr), 1e-14), 0.0, 1.0)

    parts = {
        "n_state": len(state_idx),
        "n_ret": len(ret_idx),
        "state_idx": state_idx,
        "ret_idx": ret_idx,
        "state_names": tuple(variable_names[i] for i in state_idx),
        "ret_names": tuple(variable_names[i] for i in ret_idx),
        "z_bar_state": z_bar_state,
        "z_bar_ret": z_bar_ret,
        "Phi_0_state": Phi_0_state,
        "Phi_11": Phi_11,
        "Phi_0_ret": Phi_0_ret,
        "Phi_21": Phi_21,
        "Sigma_ss": Sigma_ss,
        "Sigma_rr": Sigma_rr,
        "Sigma_rs": Sigma_rs,
        "M": M,
        "Sigma_r_cond": Sigma_r_cond,
        "Phi_12_norm": Phi_12_norm,
        "Phi_22_norm": Phi_22_norm,
        "var_explained_share": explained_share,
    }

    if verbose:
        print("=" * 64)
        print("VAR PARTITION SUMMARY")
        print("=" * 64)
        print(f"Full variables: {list(variable_names)}")
        print(f"State variables (on grid): {list(parts['state_names'])}")
        print(f"Return variables (integrated): {list(parts['ret_names'])}")
        print()

        # Stationarity of slow-state sub-VAR
        eigs = np.sort(np.abs(np.linalg.eigvals(Phi_11)))[::-1]
        max_eig = eigs[0]
        stability = "STATIONARY" if max_eig < 1.0 else "*** NON-STATIONARY ***"
        print("Slow-state sub-VAR:")
        print(f"  Phi_11 eigenvalues (|.|): {eigs.round(4).tolist()}")
        print(f"  Max |eigenvalue| = {max_eig:.4f}  --  {stability}")
        print()

        # Restriction check
        print("Restriction check (should be near zero if imposed):")
        print(f"  ||Phi_12|| = {Phi_12_norm:.6e}   (lagged returns -> states)")
        print(f"  ||Phi_22|| = {Phi_22_norm:.6e}   (lagged returns -> returns)")
        print()

        # Return variance explained
        print("Conditional return variance explained by slow-state conditioning:")
        for k, name in enumerate(parts["ret_names"]):
            print(f"  {name}: {100.0 * explained_share[k]:6.2f}%  explained"
                  f"  (residual var = {np.diag(Sigma_r_cond)[k]:.6f})")
        print("=" * 64)

    return parts

In [17]:
# =============================================================================
# 4. DISCRETIZATION FUNCTIONS
# =============================================================================

def rouwenhorst_univariate(N, mu, rho, sigma):
    """Rouwenhorst discretization for AR(1): y' = mu + rho*(y-mu) + sigma*eps."""
    if N < 2:
        raise ValueError("Rouwenhorst N must be >= 2")

    p = (1.0 + rho) / 2.0
    q = p
    Pi = np.array([[p, 1.0 - p], [1.0 - q, q]], dtype=float)

    for n in range(3, N + 1):
        Pi_new = np.zeros((n, n), dtype=float)
        Pi_new[:-1, :-1] += p * Pi
        Pi_new[:-1, 1:] += (1.0 - p) * Pi
        Pi_new[1:, :-1] += (1.0 - q) * Pi
        Pi_new[1:, 1:] += q * Pi
        Pi_new[1:-1, :] *= 0.5
        Pi = Pi_new

    sigma_y = sigma / np.sqrt(max(1e-14, 1.0 - rho * rho))
    psi = sigma_y * np.sqrt(N - 1.0)
    y_grid = np.linspace(mu - psi, mu + psi, N)

    return y_grid, Pi


def rouwenhorst_multivariate(N_vec, mu, Phi, Sigma, method="independent"):
    """
    Multivariate Rouwenhorst with independence approximation across dimensions.

    Parameters:
        N_vec: list of grid sizes per state variable
        mu: intercept in z' = mu + Phi z + eps
        Phi: persistence matrix
        Sigma: Cholesky factor of eps covariance
    """
    if method != "independent":
        raise NotImplementedError("Only method='independent' is currently implemented")

    N_vec = np.asarray(N_vec, dtype=int)
    k = len(N_vec)
    if Phi.shape != (k, k):
        raise ValueError(f"Phi must have shape {(k, k)}, got {Phi.shape}")
    if Sigma.shape != (k, k):
        raise ValueError(f"Sigma must have shape {(k, k)}, got {Sigma.shape}")

    mu_bar = np.linalg.solve(np.eye(k) - Phi, mu)
    Omega = Sigma @ Sigma.T

    grids = []
    marginals = []
    for i in range(k):
        rho_i = Phi[i, i]
        sigma_i = np.sqrt(max(1e-14, Omega[i, i]))
        g_i, Pi_i = rouwenhorst_univariate(int(N_vec[i]), mu_bar[i], rho_i, sigma_i)
        grids.append(g_i)
        marginals.append(Pi_i)

    n_total = int(np.prod(N_vec))
    state_indices = np.zeros((n_total, k), dtype=np.int64)
    for idx, multi_idx in enumerate(np.ndindex(*N_vec.tolist())):
        state_indices[idx, :] = np.array(multi_idx, dtype=np.int64)

    Pi_joint = np.ones((n_total, n_total), dtype=float)
    for dim in range(k):
        Pi_dim = marginals[dim]
        from_idx = state_indices[:, dim]
        to_idx = state_indices[:, dim]
        Pi_joint *= Pi_dim[np.ix_(from_idx, to_idx)]

    row_sums = Pi_joint.sum(axis=1, keepdims=True)
    Pi_joint = Pi_joint / np.maximum(row_sums, 1e-300)

    return grids, Pi_joint, state_indices


def discretize_income_ar1_mixture(rho, p, mu1, sigma1, mu2, sigma2, N, n_stds=3):
    """Discretize persistent income AR(1) with mixture-normal innovations."""
    mu_eta = p * mu1 + (1.0 - p) * mu2
    var_eta = p * (sigma1 ** 2 + (mu1 - mu_eta) ** 2) + (1.0 - p) * (sigma2 ** 2 + (mu2 - mu_eta) ** 2)
    std_z = np.sqrt(var_eta / max(1e-14, 1.0 - rho ** 2))

    z_grid = np.linspace(-n_stds * std_z, n_stds * std_z, N)
    dz = z_grid[1] - z_grid[0]
    half_bin = 0.5 * dz

    Pi_z = np.zeros((N, N), dtype=float)
    for i, z_t in enumerate(z_grid):
        mean_next = rho * z_t
        for j, z_next in enumerate(z_grid):
            upper = z_next + half_bin - mean_next
            lower = z_next - half_bin - mean_next
            Pi_z[i, j] = mixture_cdf(upper, p, mu1, sigma1, mu2, sigma2) - mixture_cdf(lower, p, mu1, sigma1, mu2, sigma2)
        Pi_z[i, :] /= np.maximum(Pi_z[i, :].sum(), 1e-300)

    return z_grid, Pi_z


def get_eps_quadrature_corrected(model, n_nodes=3):
    """Transitory shock quadrature using Gauss-Hermite with zero-mean enforcement.

    NOTE: model.mu_eps2 is NOT used. Component 2's mean is computed internally
    to enforce E[eps] = 0:  mu_eps2_effective = -(pe/(1-pe)) * mu_eps1.
    Only model.sigma_eps2 is used from the component-2 parameters.
    """
    nodes, weights = roots_hermite(n_nodes)
    weights = weights / np.sqrt(np.pi)
    nodes = nodes * np.sqrt(2.0)

    e1 = nodes * model.sigma_eps1 + model.mu_eps1

    mu_eps2_normalized = -(model.pe / (1.0 - model.pe)) * model.mu_eps1
    e2 = nodes * model.sigma_eps2 + mu_eps2_normalized

    w1 = weights * model.pe
    w2 = weights * (1.0 - model.pe)

    eps_nodes = np.concatenate([e1, e2])
    eps_weights = np.concatenate([w1, w2])

    mean_check = np.sum(eps_nodes * eps_weights)
    if abs(mean_check) > 1e-10:
        print(f"WARNING: transitory shock mean = {mean_check:.6e} (should be near 0)")

    return eps_nodes, eps_weights

In [18]:
# =============================================================================
# 5. PRECOMPUTE CLASS (GENERIC STATE/RETURN VERSION)
# =============================================================================

class Precompute:
    """
    Precompute grids, transitions, and lookup tables.

    Generic design:
    - Grid only over model state variables.
    - Return variables integrated via conditional means mu_r[i, j, k].

    Grid-size choice belongs here, not in LifecyclePortfolioModel.
    LifecyclePortfolioModel holds economic parameters; Precompute holds
    numerical approximation parameters. Use state_grid_sizes to control
    the trade-off between accuracy and computation time:
      - [5, 5, 5]   = 125 states, fast, coarser approximation
      - [7, 7, 7]   = 343 states, good default for production runs
      - [9, 9, 9]   = 729 states, finer, check memory (Pi_state: N_s^2 floats)
    Different sizes per dimension are allowed, e.g. [9, 7, 7] if one
    state variable has higher persistence and needs more resolution.
    The consistency check below reports the approximation error so you
    can verify the chosen grid is adequate.

    Solver input reference (all arrays consumed by Part 2):
    -------------------------------------------------------
    Grids:
      wealth_grid  (n_w,)               cash-on-hand interpolation points [geom]
      s_grid       (n_s,)               savings grid for EGM endogenous gridpoints
      ages         (n_age,)             integer ages; index t -> ages[t]

    Financial state (VAR):
      state_grid   (N_state, n_state)   joint state grid; row i = slow-state vector
      Pi_state     (N_state, N_state)   Pi_state[i,j] = P(s_{t+1}=j | s_t=i)
      mu_r         (N_state, N_state, n_ret)
                                        mu_r[i,j,0] = E[xr    | s_t=i, s_{t+1}=j]  log excess stock return
                                        mu_r[i,j,1] = E[xtips | s_t=i, s_{t+1}=j]  log excess TIPS return
      r_bill_grid  (N_state,)           log real bill rate at each slow state;
                                        R_bill = exp(r_bill_grid[i_s])  (known at decision time)

    Income:
      z_grid       (n_z,)               persistent income states (log, mean-zero)
      Pi_z         (n_z, n_z)           Pi_z[i,j] = P(z_{t+1}=j | z_t=i)
      eps_nodes    (n_eps,)             Gauss-Hermite nodes for transitory shock eps
      eps_weights  (n_eps,)             quadrature weights; sum=1, E[eps]=0 enforced

    Bequest:
      annuity_factors  (N_state,)           A(r_f, b_bar) annuity factor at each state
                                            (used in bequest_utility / bequest_marginal)

    Lookup tables:
      working_income    (n_age, n_z, n_eps)
                                        working_income[t, iz, ie] = after-tax labor income
                                        at age ages[t], persistent state iz, transitory node ie
      pension_after_tax (n_age, n_z)
                                        pension_after_tax[t, iz] = after-tax Social Security
                                        benefit; constant across ages, indexed by career z

    Dimension counters: n_w, n_s, n_z, n_eps, n_age, N_state
    Backward-compat aliases: slow_grid, Pi_slow, slow_grids, slow_state_indices, N_s
    -------------------------------------------------------
    """

    def __init__(
        self,
        model,
        disc_config=None,
        verbose=True,
    ):
        # --- Config ---
        if disc_config is None:
            disc_config = DiscretizationConfig()
        self.disc_config = disc_config
        self.model = model
        self.verbose = verbose

        # --- Grids ---
        self.wealth_grid = np.geomspace(disc_config.wealth_min, disc_config.wealth_max, disc_config.n_wealth)
        self.s_grid      = np.geomspace(disc_config.savings_min, disc_config.wealth_max, disc_config.n_savings)
        self.ages        = np.arange(model.start_age, model.terminal_age + 1)

        # --- Financial state VAR discretization ---
        state_grid_sizes = list(disc_config.state_grid_sizes)
        if len(state_grid_sizes) != model.n_state:
            raise ValueError("state_grid_sizes length must equal model.n_state")
        self.state_grid_sizes = list(state_grid_sizes)

        Sigma_state_chol = np.linalg.cholesky(model.Sigma_ss)
        self.state_grids, self.Pi_state, self.state_indices = rouwenhorst_multivariate(
            N_vec=state_grid_sizes,
            mu=model.Phi_0_state,
            Phi=model.Phi_11,
            Sigma=Sigma_state_chol,
            method="independent",
        )
        # state_grids:   list[n_state] of 1-D marginal grids, state_grids[d] has shape (state_grid_sizes[d],)
        # Pi_state:      (N_state, N_state) float64 - joint transition matrix; Pi_state[i,j] = P(s_{t+1}=j|s_t=i)
        # state_indices: (N_state, n_state) int64  - multi-index into marginal grids; row i maps to state_grids

        self.state_grid = self._build_state_grid(self.state_grids, self.state_indices)
        # (N_state, n_state) float64 - flat Cartesian grid of slow-state vectors
        # row i = [rtb, y_nom, dp] values at joint state i

        self.N_state = self.state_grid.shape[0]  # int â€” total joint states = prod(state_grid_sizes)

        # Backward-compatibility aliases (used by Part 2 solver code)
        self.slow_grids         = self.state_grids    # alias for state_grids
        self.Pi_slow            = self.Pi_state        # (N_state, N_state) alias for Pi_state
        self.slow_state_indices = self.state_indices   # (N_state, n_state) alias for state_indices
        self.slow_grid          = self.state_grid      # (N_state, n_state) alias for state_grid
        self.N_s                = self.N_state         # int alias for N_state

        # --- Conditional return means and bill rate ---
        self.mu_r = self._precompute_conditional_returns()
        # (N_state, N_state, n_ret) float64
        # mu_r[i, j, 0] = E[xr    | s_t=i, s_{t+1}=j]  - log excess stock return, conditional on transition i-j
        # mu_r[i, j, 1] = E[xtips | s_t=i, s_{t+1}=j]  - log excess TIPS return,  conditional on transition i-j
        # Use exp(mu_r[i,j,k]) to get gross excess return multiplier.

        self.r_bill_grid = self.state_grid[:, model.bill_rate_index_in_state]
        # (N_state,) float64 - log real bill rate at each slow state
        # R_bill = exp(r_bill_grid[i_s]); bill rate is KNOWN at decision time (no uncertainty)

        # --- Bequest annuity factors (one per financial state) ---
        # A(y_nom, b_bar): PV of b_bar annual payments of 1 discounted at the
        # 10-year nominal bond yield.  Using y_nom is coherent because the
        # bequest horizon b_bar equals the bond maturity: the nominal bond is
        # the natural pricing instrument for the heir's consumption stream.
        # y_nom is stored in quarterly decimal (SVENY10/400); multiply by 4
        # to recover the annual yield used for discounting.
        _y_ann = self.state_grid[:, model.annuity_yield_index_in_state] * 4.0   # quarterly -> annual yield
        self.annuity_factors = annuity_factor(_y_ann, model.b_bar)
        # (N_state,) float64 - A(y_nom, b_bar) for each financial state
        # Used by bequest_utility / bequest_marginal / bequest_marginal_inv in solver.

        # --- Income discretization ---
        self.z_grid, self.Pi_z = discretize_income_ar1_mixture(
            rho=model.rho,
            p=model.pz,
            mu1=model.mu_eta1,
            sigma1=model.sigma_eta1,
            mu2=model.mu_eta2,
            sigma2=model.sigma_eta2,
            N=disc_config.n_z,
        )
        # z_grid: (n_z,) float64 - persistent income states (log deviation from mean, mean-zero by construction)
        # Pi_z:   (n_z, n_z) float64 - Pi_z[iz, jz] = P(z_{t+1}=jz | z_t=iz)

        self.eps_nodes, self.eps_weights = get_eps_quadrature_corrected(model, n_nodes=disc_config.n_eps_nodes)
        # eps_nodes:   (n_eps,) float64 - Gauss-Hermite quadrature nodes for transitory income shock eps
        # eps_weights: (n_eps,) float64 - quadrature weights; sum(eps_weights) = 1,  E[eps] = 0 enforced

        # --- Income lookup tables ---
        self.working_income = self._precompute_working_income()
        # (n_age, n_z, n_eps) float64
        # working_income[t, iz, ie] = after-tax net labor income
        #   at age ages[t], persistent state z_grid[iz], transitory shock eps_nodes[ie]
        # Gross income: Y = exp(f(age) + z_grid[iz] + eps_nodes[ie]);  net = disposable_income_working(Y)

        self.pension_after_tax = self._precompute_pension()
        # (n_age, n_z) float64
        # pension_after_tax[t, iz] = after-tax Social Security pension benefit
        #   given career rank exp(z_grid[iz]); constant across retirement ages (same row repeated)

        # --- Dimension counters ---
        self.n_w   = len(self.wealth_grid)  # int â€” wealth grid points
        self.n_s   = len(self.s_grid)       # int â€” savings grid points
        self.n_z   = len(self.z_grid)       # int â€” persistent income states
        self.n_eps = len(self.eps_nodes)    # int â€” transitory shock quadrature nodes (= 2 * n_eps_nodes)
        self.n_age = len(self.ages)         # int â€” number of age periods

        # --- Earnings-dependent mortality (Catherine 2025, eq. 35) ---
        self.survival_probs_2d, self._chi_vec, self._mortality_diag = calibrate_earnings_dependent_mortality(
            start_age=model.start_age,
            terminal_age=model.terminal_age,
            z_grid=self.z_grid,
            rho=model.rho,
            pz=model.pz,
            mu_eta1=model.mu_eta1,
            sigma_eta1=model.sigma_eta1,
            mu_eta2=model.mu_eta2,
            sigma_eta2=model.sigma_eta2,
            verbose=self.verbose,
        )
        # survival_probs_2d: (n_age, n_z) float64
        # survival_probs_2d[t, iz] = 1 - min(chi[iz] * m_baseline(age_t), 1)

        # Diagnostics
        self._validate_conditional_returns(
            tol_warn=disc_config.consistency_tol_warn,
            tol_error=disc_config.consistency_tol_error,
        )
        if self.verbose:
            self._print_summary()

    @staticmethod
    def _build_state_grid(state_grids, state_indices):
        n_total, n_dim = state_indices.shape
        out = np.empty((n_total, n_dim), dtype=float)
        for i in range(n_total):
            for d in range(n_dim):
                out[i, d] = state_grids[d][state_indices[i, d]]
        return out

    def _precompute_conditional_returns(self):
        """
        mu_r[i, j, :] = E[r_{t+1} | state_t=i, state_{t+1}=j]

        Derived from the conditional formula:
          mu_r[i,j] = Phi_0_ret + Phi_21 @ s_i + M @ (s_j - Phi_0_state - Phi_11 @ s_i)

        Rearranged into a sum of three independent terms:
          mu_r[i,j] = const + A @ s_i + M @ s_j
          where  const = Phi_0_ret - M @ Phi_0_state       (n_ret,)
                 A     = Phi_21    - M @ Phi_11             (n_ret, n_state)

        This vectorized form avoids O(N_state^2) Python loops.
        """
        const  = self.model.Phi_0_ret - self.model.M @ self.model.Phi_0_state  # (n_ret,)
        A      = self.model.Phi_21    - self.model.M @ self.model.Phi_11        # (n_ret, n_state)

        term_i = self.state_grid @ A.T             # (N_state, n_ret)
        term_j = self.state_grid @ self.model.M.T  # (N_state, n_ret)

        return const[None, None, :] + term_i[:, None, :] + term_j[None, :, :]

    def _validate_conditional_returns(self, tol_warn=2e-2, tol_error=1e-1):
        """
        Verify sum_j Pi[i,j] * mu_r[i,j] == Phi_0_ret + Phi_21 @ s_i for all i.

        Theoretical error = M @ Phi_11_off @ s_i, where
        Phi_11_off = Phi_11 - diag(diag(Phi_11)).

        Source: independence Rouwenhorst uses only diagonal(Phi_11) per marginal,
        so it cannot match E[s_{t+1}|s_i] when Phi_11 has off-diagonal elements.
        The error grows linearly with the off-diagonal cross-persistence and with
        how far each state is from its mean (worst at grid extremes).
        Finer grids do NOT reduce this error; it is a structural approximation.
        """
        N = self.N_state
        n_ret = self.model.n_ret

        errors = np.empty((N, n_ret))
        for i in range(N):
            target = self.model.Phi_0_ret + self.model.Phi_21 @ self.state_grid[i]
            avg    = self.Pi_state[i, :] @ self.mu_r[i, :, :]
            errors[i, :] = np.abs(avg - target)

        max_err_per_ret  = errors.max(axis=0)
        mean_err_per_ret = errors.mean(axis=0)
        overall_max      = errors.max()
        worst_i          = errors.max(axis=1).argmax()

        Phi_11_off = self.model.Phi_11 - np.diag(np.diag(self.model.Phi_11))

        if self.verbose:
            print("=" * 64)
            print("CONDITIONAL RETURN CONSISTENCY CHECK")
            print("=" * 64)
            print("Error source: independence Rouwenhorst uses only diag(Phi_11).")
            print("Theoretical error at state i = M @ Phi_11_off @ s_i, where")
            print("Phi_11_off = Phi_11 - diag(diag(Phi_11)).")
            print("Error is worst at grid extremes; finer grids do not fix it.")
            print()
            print(f"  ||Phi_11_off||_F = {np.linalg.norm(Phi_11_off):.4f}"
                  "  (cross-persistence magnitude)")
            print(f"  ||M||_F          = {np.linalg.norm(self.model.M):.4f}"
                  "  (return-state conditioning strength)")
            print(f"  Product ||M @ Phi_11_off||_F = "
                  f"{np.linalg.norm(self.model.M @ Phi_11_off):.4f}"
                  "  (amplification factor)")
            print()
            print(f"  {'Variable':<12}  {'max error':>10}  {'mean error':>10}")
            print(f"  {'-'*12}  {'-'*10}  {'-'*10}")
            for k, name in enumerate(self.model.ret_names):
                flag = "  << WARN" if max_err_per_ret[k] > tol_warn else ""
                print(f"  {name:<12}  {max_err_per_ret[k]:>10.3e}"
                      f"  {mean_err_per_ret[k]:>10.3e}{flag}")
            print()
            worst_vals = "  ".join(
                f"{name}={self.state_grid[worst_i, d]:.4f}"
                for d, name in enumerate(self.model.state_names)
            )
            print(f"  Worst state: index={worst_i}  ({worst_vals})")
            print(f"  Overall max error: {overall_max:.3e}")
            print()
            if overall_max > tol_error:
                print(f"  STATUS: FAIL  (max {overall_max:.3e} > hard limit {tol_error:.3e})")
                print(f"  Current grid: {self.state_grid_sizes}  ->  {self.N_state} states")
                print("  Off-diagonal Phi_11 is too large for independence Rouwenhorst.")
                print("  See ||M @ Phi_11_off|| above for the amplified error magnitude.")
            elif overall_max > tol_warn:
                print(f"  STATUS: WARN  (max {overall_max:.3e} > soft limit {tol_warn:.3e})")
                print(f"  Current grid: {self.state_grid_sizes}  ->  {self.N_state} states")
            else:
                print(f"  STATUS: PASS  (max {overall_max:.3e} < warn limit {tol_warn:.3e})")
            print("=" * 64)

        if overall_max > tol_error:
            raise RuntimeError(
                f"Conditional return consistency error {overall_max:.3e} exceeds "
                f"hard limit {tol_error:.3e}. See printed diagnostics above."
            )

    def _precompute_working_income(self):
        """After-tax labor income table: [age, z_state, eps_node]."""
        n_age = len(self.ages)
        n_z = len(self.z_grid)
        n_eps = len(self.eps_nodes)

        out = np.empty((n_age, n_z, n_eps), dtype=float)
        for t_idx, age in enumerate(self.ages):
            det = (self.model.b0 + self.model.b1 * age
                   + self.model.b2 * (age ** 2) / 10.0
                   + self.model.b3 * (age ** 3) / 100.0)

            for iz in range(n_z):
                p_val = self.z_grid[iz]
                for ie in range(n_eps):
                    y_gross = np.exp(det + p_val + self.eps_nodes[ie])
                    out[t_idx, iz, ie] = disposable_income_working(y_gross)

        return out

    def _precompute_pension(self):
        """After-tax pension table: [age, z_state]."""
        base_pension = compute_pension_after_tax(self.z_grid)
        n_age = len(self.ages)
        n_z = len(self.z_grid)
        out = np.empty((n_age, n_z), dtype=float)
        for t_idx in range(n_age):
            out[t_idx, :] = base_pension
        return out

    def regenerate_savings_grid(self, n_s_points):
        """Utility for sensitivity runs in Part 2."""
        return np.geomspace(self.disc_config.savings_min, self.wealth_grid[-1], int(n_s_points))

    def _print_summary(self):
        print("=" * 64)
        print("PRECOMPUTE SUMMARY")
        print("=" * 64)
        sizes_str = " x ".join(str(n) for n in self.state_grid_sizes)
        print(f"Ages         : {self.model.start_age} to {self.model.terminal_age}"
              f"  ({self.n_age} periods,"
              f" retire at {self.model.retire_age})")
        print(f"State grid   : {sizes_str} = {self.N_state} joint states")
        print(f"  state vars : {list(self.model.state_names)}")
        print(f"  return vars: {list(self.model.ret_names)}")
        print(f"Income grid  : {self.n_z} persistent states"
              f"  x  {self.n_eps} transitory nodes")
        print(f"mu_r         : {self.mu_r.shape}"
              f"  ({self.N_state * self.N_state * self.model.n_ret:,} values)")
        print(f"Bill-rate idx: {self.model.bill_rate_index_in_state}"
              f"  ({self.model.state_names[self.model.bill_rate_index_in_state]})")
        print(f"r_bill range : [{self.r_bill_grid.min():.4f},"
              f" {self.r_bill_grid.max():.4f}]")
        print(f"annuity_factors   : {self.annuity_factors.shape}  range=[{self.annuity_factors.min():.2f}, {self.annuity_factors.max():.2f}]")
        print(f"working_income    : {self.working_income.shape}  (n_age x n_z x n_eps)")
        print(f"pension_after_tax : {self.pension_after_tax.shape}  (n_age x n_z)")
        print("=" * 64)


In [19]:
# =============================================================================
# 6. MODEL FACTORY (CONFIG DRIVEN)
# =============================================================================

def build_model(base_config, var_config, verbose=True):
    """Build LifecyclePortfolioModel from primitive configs."""
    u, u_prime, u_prime_inv = create_utility_functions(base_config["gamma"])

    parts = partition_var(
        Phi_full=np.asarray(var_config["Phi"], dtype=float),
        Omega_full=np.asarray(var_config["Omega"], dtype=float),
        z_bar=np.asarray(var_config["z_bar"], dtype=float),
        state_idx=var_config["state_indices"],
        ret_idx=var_config["return_indices"],
        variable_names=var_config["variable_names"],
        verbose=verbose,
    )

    bill_rate_index_in_state = int(var_config["bill_rate_index_in_state"])
    if bill_rate_index_in_state < 0 or bill_rate_index_in_state >= parts["n_state"]:
        raise ValueError("bill_rate_index_in_state is out of bounds for state vector")

    annuity_yield_index_in_state = int(var_config["annuity_yield_index_in_state"])
    if annuity_yield_index_in_state < 0 or annuity_yield_index_in_state >= parts["n_state"]:
        raise ValueError("annuity_yield_index_in_state is out of bounds for state vector")

    return LifecyclePortfolioModel(
        u=u,
        u_prime=u_prime,
        u_prime_inv=u_prime_inv,
        beta=float(base_config["beta"]),
        gamma=float(base_config["gamma"]),
        b_bar=int(base_config["b_bar"]),
        start_age=int(base_config["start_age"]),
        retire_age=int(base_config["retire_age"]),
        terminal_age=int(base_config["terminal_age"]),
        b0=float(base_config["b0"]),
        b1=float(base_config["b1"]),
        b2=float(base_config["b2"]),
        b3=float(base_config["b3"]),
        rho=float(base_config["rho"]),
        pz=float(base_config["pz"]),
        mu_eta1=float(base_config["mu_eta1"]),
        sigma_eta1=float(base_config["sigma_eta1"]),
        mu_eta2=float(base_config["mu_eta2"]),
        sigma_eta2=float(base_config["sigma_eta2"]),
        pe=float(base_config["pe"]),
        mu_eps1=float(base_config["mu_eps1"]),
        sigma_eps1=float(base_config["sigma_eps1"]),
        mu_eps2=float(base_config["mu_eps2"]),
        sigma_eps2=float(base_config["sigma_eps2"]),
        n_state=parts["n_state"],
        n_ret=parts["n_ret"],
        state_names=parts["state_names"],
        ret_names=parts["ret_names"],
        z_bar_state=parts["z_bar_state"],
        z_bar_ret=parts["z_bar_ret"],
        Phi_0_state=parts["Phi_0_state"],
        Phi_11=parts["Phi_11"],
        Phi_0_ret=parts["Phi_0_ret"],
        Phi_21=parts["Phi_21"],
        Sigma_ss=parts["Sigma_ss"],
        Sigma_rr=parts["Sigma_rr"],
        Sigma_rs=parts["Sigma_rs"],
        M=parts["M"],
        Sigma_r_cond=parts["Sigma_r_cond"],
        bill_rate_index_in_state=bill_rate_index_in_state,
        annuity_yield_index_in_state=annuity_yield_index_in_state,
        constrained=bool(base_config.get("constrained", True)),
    )


# =============================================================================
# 6B. BASE CONFIG DEFAULTS
# =============================================================================

def build_base_config_legacy_defaults():
    """Baseline non-return calibration."""
    gamma = 3.0
    beta = 0.96
    b_bar = 10          # bequest horizon in years (Catherine 2025)

    start_age = 22
    retire_age = 67
    terminal_age = 99

    rho = 0.991
    pz = 0.176
    mu_eta1, sigma_eta1 = -0.524, 0.113
    mu_eta2 = -(pz / (1.0 - pz)) * mu_eta1   # zero-mean condition
    sigma_eta2 = 0.046
    pe = 0.044
    mu_eps1, sigma_eps1 = 0.134, 0.762
    mu_eps2, sigma_eps2 = 0.0, 0.055   # NOTE: mu_eps2 overridden in get_eps_quadrature_corrected
    # Catherine (2025) age-earnings profile: b0 + b1*Age + b2*Age^2/10 + b3*Age^3/100
    b0, b1, b2, b3 = -6.142, 0.3040, -0.051, 0.002586

    return {
        "beta": beta, "gamma": gamma, "b_bar": b_bar,
        "start_age": start_age, "retire_age": retire_age, "terminal_age": terminal_age,
        "b0": b0, "b1": b1, "b2": b2, "b3": b3, "rho": rho, "pz": pz,
        "mu_eta1": mu_eta1, "sigma_eta1": sigma_eta1,
        "mu_eta2": mu_eta2, "sigma_eta2": sigma_eta2,
        "pe": pe,
        "mu_eps1": mu_eps1, "sigma_eps1": sigma_eps1,
        "mu_eps2": mu_eps2, "sigma_eps2": sigma_eps2,
        "constrained": True,
    }


# =============================================================================
# 7. VAR ESTIMATION (RESTRICTED OR UNRESTRICTED)
# =============================================================================

def _load_var_dataset(csv_path, columns):
    import pandas as pd

    df = pd.read_csv(csv_path)
    if "date" in df.columns:
        try:
            df["date"] = pd.to_datetime(df["date"])
            df = df.set_index("date")
        except Exception:
            pass

    missing_cols = [c for c in columns if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {csv_path}: {missing_cols}")

    data = df[columns].dropna().astype(float)
    if len(data) < 10:
        raise ValueError("Too few observations after dropna")

    return data


def _compute_r2_per_equation(y_true, y_hat, columns):
    out = {}
    for i, col in enumerate(columns):
        resid = y_true[:, i] - y_hat[:, i]
        sse = float(np.sum(resid ** 2))
        centered = y_true[:, i] - np.mean(y_true[:, i])
        sst = float(np.sum(centered ** 2))
        out[col] = 1.0 - sse / max(sst, 1e-14)
    return out


def _safe_residual_correlation(resid):
    corr = np.corrcoef(resid, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def estimate_var1_from_csv(csv_path, columns, trend="c"):
    """
    Estimate unrestricted VAR(1) from CSV using all lagged variables.

    Returns:
      var_core: dict with z_bar, Phi, Omega, variable_names, const,
                residual_correlation, equation_r2, estimation
      res: statsmodels VARResults
      data: estimation sample DataFrame
    """
    from statsmodels.tsa.api import VAR

    data = _load_var_dataset(csv_path=csv_path, columns=columns)
    res = VAR(data).fit(maxlags=1, ic=None, trend=trend)

    Phi = np.asarray(res.coefs[0], dtype=float)
    Omega = np.asarray(res.sigma_u, dtype=float)

    if "const" in res.params.index:
        const = np.asarray(res.params.loc["const"], dtype=float)
    else:
        const = np.zeros(len(columns), dtype=float)

    z_bar = np.linalg.solve(np.eye(len(columns)) - Phi, const)

    y_true = data.iloc[1:, :].to_numpy()
    y_hat = res.fittedvalues.to_numpy()
    resid = res.resid.to_numpy()

    var_core = {
        "z_bar": z_bar,
        "Phi": Phi,
        "Omega": Omega,
        "variable_names": list(columns),
        "const": const,
        "residual_correlation": _safe_residual_correlation(resid),
        "equation_r2": _compute_r2_per_equation(y_true, y_hat, columns),
        "estimation": "unrestricted",
        "trend": trend,
    }

    return var_core, res, data


def estimate_restricted_var1_from_csv(csv_path, columns, state_indices, trend="c"):
    """
    Estimate restricted VAR(1) where lagged return variables are excluded.

    Restriction: only lagged state variables enter each equation.
    This implies return-lag columns in Phi are zero by construction.

    Returns:
      var_core: dict with z_bar, Phi, Omega, variable_names, const,
                residual_correlation, equation_r2, estimation
      fit_details: dict with coefficient table and residuals
      data: estimation sample DataFrame
    """
    import pandas as pd

    data = _load_var_dataset(csv_path=csv_path, columns=columns)

    state_indices = np.asarray(state_indices, dtype=int)
    n = len(columns)
    if np.any(state_indices < 0) or np.any(state_indices >= n):
        raise ValueError("state_indices contains out-of-bounds index")
    if len(np.unique(state_indices)) != len(state_indices):
        raise ValueError("state_indices contains duplicates")

    state_cols = [columns[i] for i in state_indices]

    Y = data.iloc[1:, :].to_numpy()                 # z_{t+1}
    X_state = data[state_cols].iloc[:-1, :].to_numpy()  # lagged states z_t[state]
    T_eff = Y.shape[0]

    if trend == "c":
        X = np.column_stack([np.ones(T_eff), X_state])
        regressor_names = ["const"] + state_cols
    else:
        X = X_state
        regressor_names = state_cols

    # Multivariate OLS via least squares (equation-by-equation equivalent)
    # coeffs shape: (n_regressors, n_equations)
    coeffs, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    Y_hat = X @ coeffs
    resid = Y - Y_hat

    k_params = X.shape[1]
    dof = T_eff - k_params
    if dof <= 0:
        raise ValueError("Not enough observations for restricted VAR estimation")

    Omega = (resid.T @ resid) / dof

    if trend == "c":
        const = coeffs[0, :]
        slope_mat = coeffs[1:, :]   # rows correspond to state_cols
    else:
        const = np.zeros(n)
        slope_mat = coeffs

    Phi = np.zeros((n, n), dtype=float)
    for k, j in enumerate(state_indices):
        Phi[:, j] = slope_mat[k, :]

    z_bar = np.linalg.solve(np.eye(n) - Phi, const)

    coef_table = pd.DataFrame(coeffs, index=regressor_names, columns=columns)

    var_core = {
        "z_bar": z_bar,
        "Phi": Phi,
        "Omega": Omega,
        "variable_names": list(columns),
        "const": const,
        "residual_correlation": _safe_residual_correlation(resid),
        "equation_r2": _compute_r2_per_equation(Y, Y_hat, columns),
        "estimation": "restricted",
        "trend": trend,
        "state_predictor_columns": state_cols,
    }

    fit_details = {
        "coeff_table": coef_table,
        "residuals": resid,
        "n_obs_effective": T_eff,
        "dof": dof,
    }

    return var_core, fit_details, data


def build_var_config_from_dataset(
    csv_path,
    columns,
    state_indices,
    return_indices,
    bill_rate_index_in_state,
    annuity_yield_index_in_state,
    trend="c",
    estimation="restricted",
):
    """
    Estimate VAR(1) and package full var_config for build_model().

    estimation:
      - "restricted": lagged return variables excluded from all equations
      - "unrestricted": standard full VAR(1)
    """
    if estimation == "restricted":
        var_core, fit_obj, data = estimate_restricted_var1_from_csv(
            csv_path=csv_path,
            columns=columns,
            state_indices=state_indices,
            trend=trend,
        )
    elif estimation == "unrestricted":
        var_core, fit_obj, data = estimate_var1_from_csv(
            csv_path=csv_path,
            columns=columns,
            trend=trend,
        )
    else:
        raise ValueError("estimation must be either 'restricted' or 'unrestricted'")

    var_config = {
        **var_core,
        "state_indices": list(state_indices),
        "return_indices": list(return_indices),
        "bill_rate_index_in_state": int(bill_rate_index_in_state),
        "annuity_yield_index_in_state": int(annuity_yield_index_in_state),
    }

    ret_idx_arr = np.asarray(return_indices, dtype=int)
    var_config["max_abs_return_lag_coeff"] = float(np.max(np.abs(var_config["Phi"][:, ret_idx_arr])))

    print("=" * 64)
    print("VAR ESTIMATION SUMMARY")
    print("=" * 64)
    print(f"Estimation mode: {var_core['estimation']}")
    if hasattr(data.index, "min") and hasattr(data.index, "max"):
        try:
            print(f"Sample: {data.index.min().date()} to {data.index.max().date()}")
        except Exception:
            print(f"Sample rows: {len(data)}")
    else:
        print(f"Sample rows: {len(data)}")
    print(f"Columns: {columns}")
    print(f"State indices: {state_indices}")
    print(f"Return indices: {return_indices}")
    print(f"VAR k={len(columns)}, T={len(data) - 1}")
    print(f"Max |Phi[:, return_lag_cols]|: {var_config['max_abs_return_lag_coeff']:.3e}")
    print()
    print("Intercept vector (const):")
    print(np.round(var_config["const"], 6))
    print()
    print("Residual correlation matrix:")
    print(np.round(var_config["residual_correlation"], 4))
    print()
    print("Equation R2:")
    for c in columns:
        print(f"  {c}: {var_config['equation_r2'][c]:.4f}")
    print("=" * 64)

    return var_config, fit_obj, data


def build_tips_system2_var_config(
    csv_path=r"TIPS_MODEL/var_dataset.csv",
    state_indices=(0, 3, 4),
    return_indices=(1, 2),
    bill_rate_index_in_state=0,
    annuity_yield_index_in_state=2,
    trend="c",
    estimation="restricted",
):
    """
    Convenience wrapper for System 2: columns = [rtb, xr, xtips, dp, y_real].

    Default uses restricted estimation to align with state/return DP architecture.
    Set estimation="unrestricted" for diagnostic comparison.
    """
    columns = ["rtb", "xr", "xtips", "dp", "y_real"]
    return build_var_config_from_dataset(
        csv_path=csv_path,
        columns=columns,
        state_indices=state_indices,
        return_indices=return_indices,
        bill_rate_index_in_state=bill_rate_index_in_state,
        annuity_yield_index_in_state=annuity_yield_index_in_state,
        trend=trend,
        estimation=estimation,
    )


# =============================================================================
# 8. TEMPLATE USAGE
# =============================================================================

def build_nominal_system1_var_config(
    csv_path=r"TIPS_MODEL/var_dataset.csv",
    state_indices=(0, 3, 4),
    return_indices=(1, 2),
    bill_rate_index_in_state=0,
    annuity_yield_index_in_state=1,
    trend="c",
    estimation="restricted",
):
    """
    Nominal bond analog of System 2.
    columns = [rtb, xr, xb, y_nom, dp]
    States: rtb(0), y_nom(3), dp(4)   Returns: xr(1), xb(2)
    Sample: ~1980 Q1 - 2025 Q4, T=183 (vs T=87 for TIPS system).

    rtb   = ex-post real bill return (TB3MS lagged / 400 - log(CPI_t/CPI_{t-1}))
    xr    = excess real stock return (RTRP log-return - rtb)
    xb    = excess nominal bond return; inflation cancels in excess returns so
            this equals the excess real bond return as well
    y_nom = 10-year nominal yield (SVENY10_t / 400)
    dp    = log dividend-price ratio (log level)

    Partition indices are identical to System 2: state_indices=[0,3,4],
    return_indices=[1,2], bill_rate_index_in_state=0.
    """
    columns = ["rtb", "xr", "xb", "y_nom", "dp"]
    return build_var_config_from_dataset(
        csv_path=csv_path,
        columns=columns,
        state_indices=state_indices,
        return_indices=return_indices,
        bill_rate_index_in_state=bill_rate_index_in_state,
        annuity_yield_index_in_state=annuity_yield_index_in_state,
        trend=trend,
        estimation=estimation,
    )


# =============================================================================
# 6C. ANNUAL VAR COMPOUNDING  (quarterly -> annual, for annual DP periods)
# =============================================================================

def annualize_var_config(var_config, h=4):
    """
    Convert a QUARTERLY var_config to an ANNUAL one (h=4 quarters per year).

    The annual DP (ages 25-80, beta=0.96, annual income profile) requires
    annual-frequency VAR parameters.  A quarterly VAR can be exactly
    compounded to annual frequency.

    IMPORTANT: The annual "return" is the SUM of h quarterly returns
    (e.g. R_annual = r_{t+1}+...+r_{t+4}), NOT the h-step-ahead return.
    This changes the return-loading matrix Phi_21 and the intercept c_r.

    Parameters
    ----------
    var_config : dict
        Quarterly var_config from build_nominal_system1_var_config() or
        build_var_config_from_dataset().
    h : int
        Number of quarters per period (default 4 = annual).

    Returns
    -------
    var_config_ann : dict
        Annual var_config ready to pass to build_model().
        z_bar, state_indices, return_indices, variable_names are preserved.

    Annual compounding formulas
    ---------------------------
    Let  n=n_var, s=state indices, r=return indices, h=4.
    Define P_k = Phi_11^k, C_k = sum_{j=0}^{k} P_j  (cumulative sums).

    State dynamics (exact):
        Phi_11_ann = Phi_11^h                                    (state AR)
        c_s_ann    = C_{h-1} @ c_s                               (state intercept)
        Omega_ss_ann = sum_{k=0}^{h-1} P_k @ Omega_ss @ P_k^T   (state innovation cov)

    Cumulative annual return (sum of h quarterly returns):
        Phi_21_ann = Phi_21 @ C_{h-1}                            (return loading)
        c_r_ann    = h*c_r + Phi_21 @ [sum_{k=1}^{h-1} C_{k-1}] @ c_s
                   (constant correction for intermediate state drift)

    Cross-covariance Cov(u_r_annual, u_s_annual):
        Omega_rs_ann = Phi_21 @ [sum_{k=1}^{h-1} C_{k-1} @ Omega_ss @ P_{h-1-k}^T]
                     + Omega_rs @ C_{h-1}^T
        where the first term comes from state noise propagating into cumulative
        returns via intermediate periods, and the second from the contemporaneous
        eps_s / eps_r correlation at each quarter.

    Return innovation variance Var(u_r_annual):
        Omega_rr_ann = Phi_21 @ [sum_{k=1}^{h-1} C_{k-1} @ Omega_ss @ C_{k-1}^T] @ Phi_21^T
                     + h*Omega_rr
                     + Phi_21 @ [sum_{k=1}^{h-1} C_{k-1}] @ Omega_sr
                     + Omega_rs @ [sum_{k=1}^{h-1} C_{k-1}]^T @ Phi_21^T

    Stationary mean z_bar: UNCHANGED (invariant to time aggregation).

    Verification: E[R_ann | z_bar_s, z_bar_s] = h*(c_r + Phi_21 @ z_bar_s)
    i.e. the mean annual return is exactly h times the mean quarterly return.
    """
    import numpy as np

    Phi_q   = np.asarray(var_config["Phi"],  dtype=float)
    Omega_q = np.asarray(var_config["Omega"], dtype=float)
    c_q     = np.asarray(var_config["const"], dtype=float)
    z_bar   = np.asarray(var_config["z_bar"], dtype=float)

    s_idx = list(var_config["state_indices"])
    r_idx = list(var_config["return_indices"])
    ns, nr = len(s_idx), len(r_idx)

    # Extract quarterly blocks
    Phi_11  = Phi_q[np.ix_(s_idx, s_idx)]
    Phi_21  = Phi_q[np.ix_(r_idx, s_idx)]
    Omega_ss = Omega_q[np.ix_(s_idx, s_idx)]
    Omega_rs = Omega_q[np.ix_(r_idx, s_idx)]
    Omega_rr = Omega_q[np.ix_(r_idx, r_idx)]
    Omega_sr = Omega_rs.T
    c_s = c_q[s_idx]
    c_r = c_q[r_idx]

    # -- Power and cumulative-sum sequences ------------------------------------
    # P[k] = Phi_11^k,  C[k] = sum_{j=0}^{k} P[j]  for k = 0..h-1
    In = np.eye(ns)
    P = [In.copy()]            # P[0] = I
    for _ in range(h - 1):
        P.append(P[-1] @ Phi_11)

    C = [In.copy()]            # C[0] = I = P[0]
    for k in range(1, h):
        C.append(C[k-1] + P[k])

    # -- Annual state dynamics -------------------------------------------------
    Phi_11_ann = P[h-1] @ Phi_11      # Phi_11^h
    c_s_ann    = C[h-1] @ c_s

    Omega_ss_ann = sum(P[k] @ Omega_ss @ P[k].T for k in range(h))

    # -- Annual return mapping -------------------------------------------------
    Phi_21_ann = Phi_21 @ C[h-1]

    # c_r_ann: constant correction for intermediate state drift
    # c_r_ann = h*c_r + Phi_21 @ (sum_{k=1}^{h-1} C[k-1]) @ c_s
    # When evaluated at z_bar_s: gives h*(c_r + Phi_21@z_bar_s) exactly.
    sum_C_for_cr = sum(C[k-1] for k in range(1, h))   # C[0]+C[1]+...+C[h-2]
    c_r_ann = h * c_r + Phi_21 @ sum_C_for_cr @ c_s

    # -- Annual cross-covariance Omega_rs -------------------------------------
    # Omega_rs_ann = Phi_21 @ [sum_{k=1}^{h-1} C[k-1] @ Omega_ss @ P[h-1-k]^T]
    #              + Omega_rs @ C[h-1]^T
    # First term: state-noise path (eps_s_{t+k} -> s_{t+k} -> r_{t+k+..} -> u_r)
    #             & (eps_s_{t+k} -> s_{4(t+1)} via P[h-1-k])
    # Second term: contemporaneous eps_s / eps_r correlation at each quarter
    inner_rs = sum(C[k-1] @ Omega_ss @ P[h-1-k].T for k in range(1, h))
    Omega_rs_ann = Phi_21 @ inner_rs + Omega_rs @ C[h-1].T

    # -- Annual return variance Omega_rr ---------------------------------------
    # Variance from state-path contributions (k=1..h-1):
    inner_rr = sum(C[k-1] @ Omega_ss @ C[k-1].T for k in range(1, h))
    # Cross terms (state eps and return eps within same quarter):
    sum_C_for_cross = sum(C[k-1] for k in range(1, h))   # same as sum_C_for_cr
    Omega_rr_ann = (
        Phi_21 @ inner_rr @ Phi_21.T
        + h * Omega_rr
        + Phi_21 @ sum_C_for_cross @ Omega_sr
        + Omega_rs @ sum_C_for_cross.T @ Phi_21.T
    )

    # -- Assemble full annual Phi, c, Omega ------------------------------------
    n = Phi_q.shape[0]
    Phi_ann   = np.zeros((n, n))
    c_ann     = np.zeros(n)
    Omega_ann = np.zeros((n, n))

    # State block
    for i, si in enumerate(s_idx):
        for j, sj in enumerate(s_idx):
            Phi_ann[si, sj]   = Phi_11_ann[i, j]
            Omega_ann[si, sj] = Omega_ss_ann[i, j]
        c_ann[si] = c_s_ann[i]

    # Return block
    for i, ri in enumerate(r_idx):
        for j, sj in enumerate(s_idx):
            Phi_ann[ri, sj]   = Phi_21_ann[i, j]
            Omega_ann[ri, sj] = Omega_rs_ann[i, j]
            Omega_ann[sj, ri] = Omega_rs_ann[i, j]   # symmetry
        for j, rj in enumerate(r_idx):
            Omega_ann[ri, rj] = Omega_rr_ann[i, j]
        c_ann[ri] = c_r_ann[i]

    # -- Verification ----------------------------------------------------------
    z_bar_s = z_bar[s_idx]
    mean_r_q = c_r + Phi_21 @ z_bar_s
    mean_r_a = c_r_ann + Phi_21_ann @ z_bar_s
    ratio = mean_r_a / mean_r_q if np.all(np.abs(mean_r_q) > 1e-12) else np.ones(nr) * h
    if not np.allclose(ratio, h, atol=1e-8):
        import warnings
        warnings.warn(
            f"annualize_var_config: mean-return ratio differs from h={h}: {ratio}",
            stacklevel=2,
        )

    var_config_ann = dict(var_config)    # shallow copy (scalar fields preserved)
    var_config_ann["Phi"]   = Phi_ann.tolist()
    var_config_ann["const"] = c_ann.tolist()
    var_config_ann["Omega"] = Omega_ann.tolist()
    # z_bar is invariant; state_indices, return_indices, variable_names unchanged
    var_config_ann["_annualized"] = True
    var_config_ann["_periods_per_annual"] = h

    print(f"annualize_var_config: compounded {h} quarters to 1 annual period.")
    print(f"  Phi_11 diagonal (annual): {np.diag(Phi_11_ann)}")
    mean_q_pct = mean_r_q * 100
    mean_a_pct = mean_r_a * 100
    ret_names = [var_config["variable_names"][i] for i in r_idx]
    for i, rn in enumerate(ret_names):
        print(f"  {rn}: quarterly mean={mean_q_pct[i]:.2f}%  annual mean={mean_a_pct[i]:.2f}%  ratio={ratio[i]:.4f}")

    return var_config_ann

# Example workflow:
# 1) Estimate nominal System 1 VAR from CSV (restricted by default):
#       var_config, fit_obj, data1 = build_nominal_system1_var_config(estimation="restricted")
#
#    Optional diagnostic comparison:
#       var_config_u, fit_u, data1_u = build_nominal_system1_var_config(estimation="unrestricted")
#
# (Legacy) Estimate TIPS System 2 VAR:
#       var_config, fit_obj, data2 = build_tips_system2_var_config(estimation="restricted")
#
# 2) Build base config from legacy defaults:
#       base_config = build_base_config_legacy_defaults()
#
# 3) Build model:
#       model = build_model(base_config, var_config, verbose=True)
#
# 4) Run precompute:
#       pc = Precompute(model, state_grid_sizes=[5, 5, 5], n_z=11, n_eps_nodes=5)


In [20]:
# =============================================================================
# 8B. HARDCODED VAR PARAMETERS  (fallback if var_dataset.csv is unavailable)
# =============================================================================
# Estimated from var_dataset.csv using build_nominal_system1_var_config()
#
#   System   : Nominal Bond (System 1)
#   Columns  : [rtb, xr, xb, y_nom, dp]
#   States   : rtb(0), y_nom(3), dp(4)   Returns: xr(1), xb(2)
#   Estimation: restricted VAR(1), lagged returns excluded from all equations
#   Omega    : from unrestricted VAR residuals (standard in state-return architecture)
#   Sample   : 1980-03-31 to 2025-12-31  T=183 quarterly observations
#   Data src : SVENY10 (GSW feds200628), TB3MS (FRED), CPI (FRED), Shiller RTRP, ie_data.xls
#
# !! IMPORTANT: VAR is estimated at QUARTERLY frequency.
# !! If the DP solver uses ANNUAL periods (ages 25-80, beta=0.96), you must
# !! annualise these matrices before passing to build_model():
# !!   Phi_annual  = Phi_quarterly @ Phi_quarterly @ Phi_quarterly @ Phi_quarterly
# !!                 (matrix power 4, or np.linalg.matrix_power(Phi, 4))
# !!   Omega_annual = sum of 4-step innovation covariance (more complex; see below)
# !! Using quarterly parameters in an annual model understates returns (by ~4x)
# !! and overstates state persistence.
# =============================================================================

import numpy as np

# Variable order: [rtb, xr, xb, y_nom, dp]
_NOM_COLS   = ["rtb", "xr", "xb", "y_nom", "dp"]
_STATE_IDX  = [0, 3, 4]   # rtb, y_nom, dp
_RET_IDX    = [1, 2]      # xr, xb

# --- Unconditional means (quarterly decimal) ---
# Annualised: rtb=-0.33%/yr  xr=+5.36%/yr  xb=+2.36%/yr  y_nom=+3.65%/yr  dp=-4.148 (log level)
_Z_BAR = np.array([
    -8.34998757e-04,   # rtb   = -0.33%/yr real bill rate
     1.33971778e-02,   # xr    = +5.36%/yr excess stock return
     5.90140636e-03,   # xb    = +2.36%/yr excess nominal bond return
     9.12159768e-03,   # y_nom = +3.65%/yr 10-year nominal yield (SVENY10)
    -4.14849497e+00,   # dp    = -4.148    log dividend-price ratio
])

# --- Intercept vector c  (quarterly) ---
_CONST = np.array([
    -1.80395400e-02,   # rtb
     3.08321600e-01,   # xr
    -3.26518200e-02,   # xb
     1.43097000e-03,   # y_nom
    -2.59384330e-01,   # dp
])

# --- AR(1) coefficient matrix Phi  (quarterly, restricted: return-lag columns = 0) ---
# Rows = z_{t+1} equations, Cols = z_t predictors
# Phi[i, j] = coefficient on lagged z_j in equation for z_i
# Return columns (1=xr, 2=xb) are zero by restriction.
#
#          L.rtb        L.xr   L.xb    L.y_nom       L.dp
_PHI = np.array([
    [ 2.50280563e-01,  0.0,    0.0,   6.64815958e-01, -2.73577227e-03],  # rtb
    [ 5.47898136e-01,  0.0,    0.0,  -3.61859878e+00,  6.30251519e-02],  # xr
    [-4.07263720e-01,  0.0,    0.0,   1.49978933e+00, -5.91363571e-03],  # xb
    [ 8.43938656e-03,  0.0,    0.0,   9.57201184e-01,  2.49133416e-04],  # y_nom
    [-1.09728419e+00,  0.0,    0.0,   2.87318397e+00,  9.44013413e-01],  # dp
])

# --- Residual covariance matrix Omega  (quarterly, from unrestricted VAR) ---
# Diagonal std devs (annualised): rtb=1.26%  xr=14.46%  xb=11.94%  y_nom=0.30%  dp=0.076
#          rtb          xr           xb           y_nom        dp
_OMEGA = np.array([
    [ 3.95595584e-05, -2.58579310e-05,  1.17037505e-04, -3.15218298e-06,  2.65077118e-05],  # rtb
    [-2.58579310e-05,  5.22952531e-03, -3.52798917e-04,  8.06645511e-06, -5.33055869e-03],  # xr
    [ 1.17037505e-04, -3.52798917e-04,  3.56188480e-03, -8.98808775e-05,  3.89367599e-04],  # xb
    [-3.15218298e-06,  8.06645511e-06, -8.98808775e-05,  2.28000460e-06, -9.06427166e-06],  # y_nom
    [ 2.65077118e-05, -5.33055869e-03,  3.89367599e-04, -9.06427166e-06,  5.73108875e-03],  # dp
])


def build_nominal_system1_var_config_hardcoded():
    """
    Fallback: return var_config using hardcoded parameter estimates.
    Use when var_dataset.csv is unavailable.
    Identical structure to build_nominal_system1_var_config() output.

    WARNING: parameters are quarterly. See frequency note above.
    """
    print("Using HARDCODED VAR parameters (nominal System 1, quarterly, T=183).")
    print("  Sample: 1980-03-31 to 2025-12-31")
    print("  !! Read frequency warning in cell 8B before use in annual DP solver.")
    return {
        "z_bar":                  _Z_BAR.copy(),
        "Phi":                    _PHI.copy(),
        "Omega":                  _OMEGA.copy(),
        "const":                  _CONST.copy(),
        "variable_names":         list(_NOM_COLS),
        "state_indices":          list(_STATE_IDX),
        "return_indices":         list(_RET_IDX),
        "bill_rate_index_in_state": 0,
        "annuity_yield_index_in_state": 1,   # y_nom is 2nd state variable [rtb, y_nom, dp]
        "max_abs_return_lag_coeff": 0.0,
        "estimation":             "restricted_hardcoded",
        "trend":                  "c",
        "state_predictor_columns": ["rtb", "y_nom", "dp"],
        "residual_correlation":   None,
        "equation_r2":            None,
    }

# =============================================================================
# 8B-ANNUAL.  HARDCODED ANNUAL VAR PARAMETERS  (derived from quarterly above)
# =============================================================================
# Derived by annualize_var_config(_CONST, _PHI, _OMEGA, h=4).
# Use these if you want a hardcoded annual fallback.
#
#   Phi_21_annual  = Phi_21 @ (I + Phi11 + Phi11^2 + Phi11^3)   -- cumulative sum
#   c_r_annual     = 4*c_r + Phi_21 @ correction @ c_s
#   Omega_rs_annual = correctly accounts for within-year state noise
#
# Mean annual returns (annualised by 4x quarterly):
#   xr = 5.36%/yr  (quarterly mean x4)
#   xb = 2.36%/yr  (quarterly mean x4)
# =============================================================================

_ANN_CONST = np.array([-0.01883853,  1.08364778, -0.08666615,  0.00454395, -0.87108414])

#          [s/r, s: rtb  y_nom   dp  |  r: xr   xb]   return cols are 0 by restriction
_ANN_PHI = np.array([[ 1.69494487e-02,  0.00000000e+00,  0.00000000e+00,
             7.75294692e-01, -2.63848727e-03],
           [ 4.07706249e-01,  0.00000000e+00,  0.00000000e+00,
            -1.15339468e+01,  2.22854466e-01],
           [-4.85835159e-01,  0.00000000e+00,  0.00000000e+00,
             4.65334874e+00, -1.62517112e-02],
           [ 9.27614826e-03,  0.00000000e+00,  0.00000000e+00,
             8.61133510e-01,  7.88120296e-04],
           [-1.19573205e+00,  0.00000000e+00,  0.00000000e+00,
             7.57961104e+00,  8.06930563e-01]])

_ANN_OMEGA = np.array([[ 4.48423138e-05,  7.41425361e-05, -7.92967578e-05,
             6.51135756e-07, -1.42390122e-05],
           [ 7.41425361e-05,  1.74769285e-02,  6.69564294e-04,
            -2.70924130e-05, -1.74774631e-02],
           [-7.92967578e-05,  6.69564294e-04,  1.25968804e-02,
            -3.13792998e-04, -3.97880400e-04],
           [ 6.51135756e-07, -2.70924130e-05, -3.13792998e-04,
             7.94891334e-06,  1.34192037e-05],
           [-1.42390122e-05, -1.74774631e-02, -3.97880400e-04,
             1.34192037e-05,  1.96116803e-02]])

def build_nominal_system1_var_config_annual_hardcoded():
    """Fallback: annual var_config using hardcoded annual parameter estimates."""
    var_config_ann = build_nominal_system1_var_config_hardcoded()
    var_config_ann["Phi"]   = _ANN_PHI.tolist()
    var_config_ann["const"] = _ANN_CONST.tolist()
    var_config_ann["Omega"] = _ANN_OMEGA.tolist()
    var_config_ann["_annualized"] = True
    var_config_ann["_periods_per_annual"] = 4
    return var_config_ann


In [21]:
# =============================================================================
# 9. DIAGNOSTIC REPORT
# =============================================================================

def print_model_diagnostic_report(model, pc, periods_per_year=4):
    """
    Comprehensive diagnostic report for calibration verification and debugging.

    Parameters
    ----------
    model : LifecyclePortfolioModel
    pc    : Precompute
    periods_per_year : int
        4 = quarterly data (default), 12 = monthly, 1 = annual.
        Used to annualize rates and scale stds from per-period to per-year.
        Note: dp (log dividend-price ratio) is a log level, not a rate;
        annualizing it via x*ppy has no economic meaning.
    """
    W   = 76
    ppy = periods_per_year
    sv  = list(model.state_names)
    rv  = list(model.ret_names)

    def header(title):
        print()
        print("=" * W)
        print(f"  {title}")
        print("=" * W)

    def sub(title):
        print(f"\n  --- {title} ---")

    def flag(label, ok, detail=""):
        tag = "[PASS]" if ok else "[WARN]"
        suf = f"  {detail}" if detail else ""
        print(f"  {tag}  {label}{suf}")

    # =========================================================================
    # 1. PREFERENCES & LIFECYCLE
    # =========================================================================
    header("1.  PREFERENCES & LIFECYCLE")

    sub("Utility & bequest")
    print(f"  gamma  = {model.gamma:.3f}   (CRRA risk aversion)")
    beta_ann = model.beta ** ppy
    print(f"  beta   = {model.beta:.6f}  per period   ->   {beta_ann:.6f}  annualized")
    print(f"  b_bar  = {model.b_bar}   (bequest horizon in years, Catherine 2025)")

    sub("Lifecycle")
    n_work = model.retire_age - model.start_age
    n_ret  = model.terminal_age - model.retire_age + 1
    print(f"  Ages:  {model.start_age} - {model.terminal_age}   ({pc.n_age} periods total)")
    print(f"  Retirement at {model.retire_age}:   {n_work} working periods,  {n_ret} retirement periods")

    sub("Survival probabilities at key ages (earnings-dependent)")
    iz_lo, iz_mid, iz_hi = 0, pc.n_z // 2, pc.n_z - 1
    print(f"  {'Age':>4}  {'low-z':>10}  {'mid-z':>10}  {'high-z':>10}")
    key_sp = [25, 30, 40, 50, 55, 60, 65, 70, 75, 80]
    for age in key_sp:
        if model.start_age <= age <= model.terminal_age:
            t = age - model.start_age
            sp_lo  = pc.survival_probs_2d[t, iz_lo]
            sp_mid = pc.survival_probs_2d[t, iz_mid]
            sp_hi  = pc.survival_probs_2d[t, iz_hi]
            print(f"  {age:>4}  {sp_lo:>10.5f}  {sp_mid:>10.5f}  {sp_hi:>10.5f}")

    # =========================================================================
    # 2. INCOME PROCESS
    # =========================================================================
    header("2.  INCOME PROCESS")

    sub("Persistent AR(1) with mixture-normal innovations")
    mu_eta  = model.pz * model.mu_eta1 + (1.0 - model.pz) * model.mu_eta2
    var_eta = (model.pz * (model.sigma_eta1**2 + (model.mu_eta1 - mu_eta)**2)
               + (1.0 - model.pz) * (model.sigma_eta2**2 + (model.mu_eta2 - mu_eta)**2))
    std_eta = np.sqrt(var_eta)
    std_z   = np.sqrt(var_eta / max(1e-14, 1.0 - model.rho**2))
    print(f"  rho        = {model.rho:.5f}  (per-period persistence)")
    print(f"  pz         = {model.pz:.3f}    (mixture weight on component 1)")
    print(f"  Component 1:  mu_eta1 = {model.mu_eta1:+.4f},  sigma_eta1 = {model.sigma_eta1:.4f}")
    print(f"  Component 2:  mu_eta2 = {model.mu_eta2:+.4f},  sigma_eta2 = {model.sigma_eta2:.4f}")
    print(f"  E[eta]     = {mu_eta:.2e}   (should be - 0)")
    print(f"  Std[eta]   = {std_eta:.5f}")
    print(f"  Std[z]     = {std_z:.5f}  (unconditional)")
    z_cover = pc.z_grid.max() / std_z if std_z > 0 else float("nan")
    print(f"  z_grid     : {pc.n_z} points  [{pc.z_grid.min():.4f}, {pc.z_grid.max():.4f}]   Â±{z_cover:.2f} Ïƒ")

    sub("Transitory shock (mixture, zero-mean enforced)")
    mu_eps2_eff = -(model.pe / (1.0 - model.pe)) * model.mu_eps1
    eps_mean    = float(np.sum(pc.eps_nodes * pc.eps_weights))
    eps_var     = float(np.sum(pc.eps_nodes**2 * pc.eps_weights))
    print(f"  pe            = {model.pe:.3f}   (probability of large-shock component)")
    print(f"  Component 1:  mu_eps1    = {model.mu_eps1:+.4f},  sigma_eps1 = {model.sigma_eps1:.4f}")
    print(f"  Component 2:  mu_eps2_eff = {mu_eps2_eff:+.4f}  (zero-mean enforced; model.mu_eps2 ignored)")
    print(f"               sigma_eps2 = {model.sigma_eps2:.4f}")
    print(f"  E[eps]        = {eps_mean:.2e}   (should be - 0)")
    print(f"  Var[eps]      = {eps_var:.5f}   Std[eps] = {np.sqrt(eps_var):.5f}")
    print(f"  eps_weights sum = {pc.eps_weights.sum():.8f}   (should be 1.000)")
    print(f"  eps_nodes     : {pc.n_eps} nodes  [{pc.eps_nodes.min():.4f}, {pc.eps_nodes.max():.4f}]")

    sub("Deterministic income profile  log Y_det = b0 + b1*Age + b2*Age^2/10 + b3*Age^3/100")
    print(f"  b0 = {model.b0:.4f},  b1 = {model.b1:.4f},  b2 = {model.b2:.4f},  b3 = {model.b3:.6f}")
    # Find peak age numerically
    _ages_det = np.arange(model.start_age, model.retire_age)
    _det_vals = model.b0 + model.b1 * _ages_det + model.b2 * _ages_det**2 / 10.0 + model.b3 * _ages_det**3 / 100.0
    _peak_idx = int(np.argmax(_det_vals))
    print(f"  Hump peak: age {_ages_det[_peak_idx]},  log-income = {_det_vals[_peak_idx]:.4f}")

    # Find grid points closest to z=0 and eps=0
    iz0 = int(np.argmin(np.abs(pc.z_grid)))
    ie0 = int(np.argmin(np.abs(pc.eps_nodes)))
    print(f"\n  Income at z - 0 (grid point {iz0}: z={pc.z_grid[iz0]:.4f}),")
    print(f"             eps - 0 (node {ie0}: eps={pc.eps_nodes[ie0]:.4f}):")
    print()
    print(f"  {'Age':>4}  {'t':>4}  {'log-det':>9}  {'Y_gross':>10}  {'Y_net (after-tax)':>18}")
    key_ages_work = [a for a in [25, 30, 35, 40, 45, 50, 55, 60, 64]
                     if model.start_age <= a < model.retire_age]
    for age in key_ages_work:
        t    = age - model.start_age
        det  = model.b0 + model.b1 * age + model.b2 * (age ** 2) / 10.0 + model.b3 * (age ** 3) / 100.0
        y_gr = np.exp(det + pc.z_grid[iz0] + pc.eps_nodes[ie0])
        y_nt = float(pc.working_income[t, iz0, ie0])
        print(f"  {age:>4}  {t:>4}  {det:>9.4f}  {y_gr:>10.4f}  {y_nt:>18.4f}")

    last_t     = model.retire_age - 1 - model.start_age
    y_last     = float(pc.working_income[last_t, iz0, ie0])
    pens_mean  = float(pc.pension_after_tax[last_t, iz0])
    repl_rate  = pens_mean / y_last if y_last > 0 else float("nan")
    print()
    print(f"  Last working year (age {model.retire_age - 1}): Y_net = {y_last:.4f}")
    print(f"  Pension at z - 0:                   {pens_mean:.4f}")
    print(f"  Replacement rate:                   {repl_rate:.2%}")
    print(f"  Pension range (min z, max z):        [{pc.pension_after_tax[0, 0]:.4f}, {pc.pension_after_tax[0, -1]:.4f}]")

    # =========================================================================
    # 3. VAR STRUCTURE
    # =========================================================================
    header(f"3.  VAR STRUCTURE   (per-period units;  Ã—{ppy} to annualize rates)")

    sub("Unconditional means")
    print(f"  State variables  (z_bar_state):")
    for d, name in enumerate(sv):
        v = model.z_bar_state[d]
        ann = f"{v * ppy * 100:+.3f}%/yr" if name not in ("dp",) else "(log level, not a rate)"
        print(f"    {name:>10}:  {v:+.8f}   annualized - {ann}")
    print(f"  Return variables  (z_bar_ret):")
    for k, name in enumerate(rv):
        v = model.z_bar_ret[k]
        print(f"    {name:>10}:  {v:+.8f}   annualized - {v * ppy * 100:+.3f}%/yr")

    # --- matrix printer ---
    def print_matrix(mat, row_names, col_names, indent="  "):
        cw = 10
        print(indent + " " * 12 + "".join(f"{c:>{cw}}" for c in col_names))
        for i, rn in enumerate(row_names):
            row = indent + f"{rn:>12}" + "".join(f"{mat[i, j]:>{cw}.5f}" for j in range(mat.shape[1]))
            print(row)

    sub("Phi_11  (state-to-state persistence)")
    print_matrix(model.Phi_11, sv, sv)
    eigs_11 = np.sort(np.abs(np.linalg.eigvals(model.Phi_11)))[::-1]
    eig_str = ", ".join(f"{e:.4f}" for e in eigs_11)
    stat    = "STATIONARY" if eigs_11[0] < 1.0 else "*** NON-STATIONARY ***"
    print(f"  Eigenvalues |Î»|: [{eig_str}]   -  {stat}")

    sub("Phi_21  (state â†’ return; return loadings on lagged state)")
    print_matrix(model.Phi_21, rv, sv)

    sub("Phi_0_ret  (return intercepts)")
    for k, name in enumerate(rv):
        v = model.Phi_0_ret[k]
        print(f"    {name:>10}:  {v:+.8f}   annualized - {v * ppy * 100:+.3f}%/yr")

    sub("M  (return | next-state conditioning,  Schur complement)")
    print_matrix(model.M, rv, sv)
    print(f"  ||M||_F  = {np.linalg.norm(model.M):.5f}")
    Phi_11_off = model.Phi_11 - np.diag(np.diag(model.Phi_11))
    print(f"  ||M @ Phi_11_off||_F = {np.linalg.norm(model.M @ Phi_11_off):.5f}"
          "  (independence-Rouwenhorst approximation error driver)")

    sub("Return standard deviations (annualized)")
    print(f"  {'Return':>8}   {'Cond. Ïƒ (given s_t,s_{t+1})':>30}   {'Uncond. Ïƒ':>18}")
    for k, name in enumerate(rv):
        cond_std  = np.sqrt(max(0.0, model.Sigma_r_cond[k, k]))
        uncond_std = np.sqrt(max(0.0, model.Sigma_rr[k, k]))
        cond_ann  = cond_std  * np.sqrt(ppy) * 100
        uncond_ann = uncond_std * np.sqrt(ppy) * 100
        print(f"  {name:>8}   {cond_ann:>25.3f}%/yr   {uncond_ann:>13.3f}%/yr")

    # =========================================================================
    # 4. STATE GRID COVERAGE
    # =========================================================================
    header("4.  STATE GRID COVERAGE")

    N_per_dim = pc.state_grid_sizes
    print(f"  Grid sizes: {N_per_dim}   -   N_state = {pc.N_state} joint states")
    print(f"  Rouwenhorst coverage per dimension: -(N-1) Ïƒ")
    print()
    print(f"  {'Var':>8}  {'N':>3}  {'- cover':>9}  {'Grid min':>10}  {'Grid max':>10}  "
          f"{'Uncond. Î¼':>11}  {'Uncond. Ïƒ':>11}  Ann. range (Ã—{ppy}Ã—100)")
    print(f"  {'-'*8}  {'-'*3}  {'-'*9}  {'-'*10}  {'-'*10}  {'-'*11}  {'-'*11}  {'-'*30}")

    for d, name in enumerate(sv):
        g      = pc.state_grids[d]
        Nd     = len(g)
        mu_d   = model.z_bar_state[d]
        rho_d  = model.Phi_11[d, d]
        # unconditional Ïƒ from the residual variance used by Rouwenhorst
        sig_inn = np.sqrt(max(1e-14, model.Sigma_ss[d, d]))
        sig_y   = sig_inn / np.sqrt(max(1e-14, 1.0 - rho_d**2))
        cover   = (g.max() - mu_d) / sig_y if sig_y > 0 else float("nan")
        ann_lo  = g.min() * ppy * 100
        ann_hi  = g.max() * ppy * 100
        print(f"  {name:>8}  {Nd:>3}  {cover:>+9.2f}  {g.min():>10.5f}  {g.max():>10.5f}  "
              f"{mu_d:>11.5f}  {sig_y:>11.5f}  [{ann_lo:.2f}%, {ann_hi:.2f}%]")

    sub("r_bill_grid  (real bill rate at each joint state,  annualized)")
    ann_lo_b = pc.r_bill_grid.min() * ppy * 100
    ann_hi_b = pc.r_bill_grid.max() * ppy * 100
    ann_mu_b = pc.r_bill_grid.mean() * ppy * 100
    print(f"  Range: [{ann_lo_b:.3f}%, {ann_hi_b:.3f}%]   Mean: {ann_mu_b:.3f}%   ({pc.N_state} values)")

    # =========================================================================
    # 5. CONDITIONAL RETURN DISTRIBUTION
    # =========================================================================
    header("5.  CONDITIONAL RETURN DISTRIBUTION")

    print(f"  mu_r shape: {pc.mu_r.shape}   (N_state Ã— N_state Ã— n_ret)")

    # Stationary distribution of Pi_state
    try:
        evals, evecs = np.linalg.eig(pc.Pi_state.T)
        idx  = int(np.argmin(np.abs(evals - 1.0)))
        stat = np.real(evecs[:, idx])
        stat = np.abs(stat) / np.abs(stat).sum()
    except Exception:
        stat = np.ones(pc.N_state) / pc.N_state

    # E[return | state_t=i] = sum_j Pi[i,j] * mu_r[i,j,k]
    E_ret_by_state = np.einsum("ij,ijk->ik", pc.Pi_state, pc.mu_r)  # (N_state, n_ret)

    print()
    print(f"  {'Return':>8}  {'mu_r min':>10}  {'mu_r max':>10}  "
          f"{'Ann. min':>12}  {'Ann. max':>12}  {'Cond.Ïƒ (ann)':>14}  {'Uncond.E[r] (ann)':>18}")
    print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*12}  {'-'*12}  {'-'*14}  {'-'*18}")
    for k, name in enumerate(rv):
        mu_lo  = float(pc.mu_r[:, :, k].min())
        mu_hi  = float(pc.mu_r[:, :, k].max())
        cs_ann = np.sqrt(max(0.0, model.Sigma_r_cond[k, k])) * np.sqrt(ppy) * 100
        e_unc  = float(stat @ E_ret_by_state[:, k]) * ppy * 100
        print(f"  {name:>8}  {mu_lo:>10.5f}  {mu_hi:>10.5f}  "
              f"{mu_lo*ppy*100:>12.3f}%  {mu_hi*ppy*100:>12.3f}%  "
              f"{cs_ann:>12.3f}%/yr  {e_unc:>16.3f}%/yr")

    sub("E[return | state_t=i]  across all states  (annualized %/yr)")
    print(f"  {'Return':>8}  {'Minimum state':>16}  {'Maximum state':>16}  {'Stationary mean':>18}")
    for k, name in enumerate(rv):
        e_min = float(E_ret_by_state[:, k].min()) * ppy * 100
        e_max = float(E_ret_by_state[:, k].max()) * ppy * 100
        e_stat = float(stat @ E_ret_by_state[:, k]) * ppy * 100
        print(f"  {name:>8}  {e_min:>13.3f}%  {e_max:>13.3f}%  {e_stat:>15.3f}%")

    # =========================================================================
    # 6. NUMERICAL SETUP & MEMORY
    # =========================================================================
    header("6.  NUMERICAL SETUP & MEMORY FOOTPRINT")

    sub("Dimension table")
    dims = [
        ("n_w",     pc.n_w,     "wealth grid points"),
        ("n_s",     pc.n_s,     "savings grid points (EGM)"),
        ("n_z",     pc.n_z,     "persistent income states"),
        ("n_eps",   pc.n_eps,   f"transitory shock nodes  (= 2 Ã— {pc.n_eps // 2} GH nodes)"),
        ("n_age",   pc.n_age,   f"age periods  ({model.start_age}â€“{model.terminal_age})"),
        ("N_state", pc.N_state, f"joint VAR states  ({'Ã—'.join(str(n) for n in pc.state_grid_sizes)})"),
        ("n_state", model.n_state, "slow-state variables"),
        ("n_ret",   model.n_ret,   "return variables (integrated)"),
    ]
    for name, val, desc in dims:
        print(f"  {name:>10} = {val:>6}   {desc}")

    sub("Memory estimate for Part 2 arrays  (float64, 8 bytes/element)")
    bpf = 8

    def mb(n): return n * bpf / 1024**2

    rows = [
        ("Value function",   pc.n_w * pc.n_z * pc.N_state * pc.n_age,
         f"n_w Ã— n_z Ã— N_state Ã— n_age  = {pc.n_w}Ã—{pc.n_z}Ã—{pc.N_state}Ã—{pc.n_age}"),
        ("Policy (Î±_s,Î±_r)", 2 * pc.n_w * pc.n_z * pc.N_state * pc.n_age,
         "2 Ã— same"),
        ("mu_r",             pc.N_state**2 * model.n_ret,
         f"N_stateÂ² Ã— n_ret  = {pc.N_state}Â²Ã—{model.n_ret}"),
        ("Pi_state",         pc.N_state**2,
         f"N_stateÂ²  = {pc.N_state}Â²"),
        ("working_income",   pc.n_age * pc.n_z * pc.n_eps,
         f"n_age Ã— n_z Ã— n_eps  = {pc.n_age}Ã—{pc.n_z}Ã—{pc.n_eps}"),
        ("pension_after_tax", pc.n_age * pc.n_z,
         f"n_age Ã— n_z  = {pc.n_age}Ã—{pc.n_z}"),
    ]
    total_el = sum(r[1] for r in rows)
    print(f"  {'Array':<22}  {'Elements':>12}  {'MB':>8}  Description")
    print(f"  {'-'*22}  {'-'*12}  {'-'*8}  {'-'*40}")
    for arr_name, n_el, desc in rows:
        print(f"  {arr_name:<22}  {n_el:>12,}  {mb(n_el):>8.2f}  {desc}")
    print(f"  {'-'*22}  {'-'*12}  {'-'*8}")
    print(f"  {'TOTAL (approx)':<22}  {total_el:>12,}  {mb(total_el):>8.2f}")

    sub("Grid ranges")
    print(f"  wealth_grid : [{pc.wealth_grid.min():.6f}, {pc.wealth_grid.max():.1f}]  (geometric,  {pc.n_w} pts)")
    print(f"  s_grid      : [{pc.s_grid.min():.2e}, {pc.s_grid.max():.1f}]  (geometric,  {pc.n_s} pts)")

    # =========================================================================
    # 7. SANITY CHECKS
    # =========================================================================
    header("7.  SANITY CHECKS")

    eig_11_max = float(np.max(np.abs(np.linalg.eigvals(model.Phi_11))))
    flag("Phi_11 stationary",     eig_11_max < 1.0,
         f"max|Î»| = {eig_11_max:.5f}")

    flag("Income innovation mean â‰ˆ 0", abs(mu_eta) < 1e-6,
         f"E[eta] = {mu_eta:.2e}")

    flag("Transitory shock mean â‰ˆ 0",  abs(eps_mean) < 1e-10,
         f"E[eps] = {eps_mean:.2e}")

    flag("eps_weights sum to 1",
         abs(pc.eps_weights.sum() - 1.0) < 1e-10,
         f"sum = {pc.eps_weights.sum():.10f}")

    pi_s_ok  = np.allclose(pc.Pi_state.sum(axis=1), 1.0, atol=1e-10)
    flag("Pi_state row sums = 1", pi_s_ok,
         f"max deviation = {np.abs(pc.Pi_state.sum(axis=1) - 1.0).max():.2e}")

    pi_z_ok  = np.allclose(pc.Pi_z.sum(axis=1), 1.0, atol=1e-10)
    flag("Pi_z row sums = 1",     pi_z_ok,
         f"max deviation = {np.abs(pc.Pi_z.sum(axis=1) - 1.0).max():.2e}")

    try:
        np.linalg.cholesky(model.Sigma_ss)
        flag("Sigma_ss positive definite",     True)
    except np.linalg.LinAlgError:
        flag("Sigma_ss positive definite",     False, "Cholesky decomposition failed")

    try:
        np.linalg.cholesky(model.Sigma_r_cond)
        flag("Sigma_r_cond positive definite", True)
    except np.linalg.LinAlgError:
        flag("Sigma_r_cond positive definite", False, "Cholesky decomposition failed")

    surv_ok = bool(np.all((pc.survival_probs_2d > 0) & (pc.survival_probs_2d <= 1.0)))
    flag("Survival probs in (0, 1]", surv_ok,
         f"range = [{pc.survival_probs_2d.min():.5f}, {pc.survival_probs_2d.max():.5f}]")

    wi_pos = bool(np.all(pc.working_income > 0))
    flag("Working income > 0 everywhere", wi_pos,
         f"min = {pc.working_income.min():.6f}")

    pens_pos = bool(np.all(pc.pension_after_tax > 0))
    flag("Pension > 0 for all z states", pens_pos,
         f"min = {pc.pension_after_tax.min():.6f}")

    flag("Wealth grid strictly positive", bool(pc.wealth_grid.min() > 0),
         f"min = {pc.wealth_grid.min():.2e}")

    # Grid coverage per state dimension
    for d, name in enumerate(sv):
        g       = pc.state_grids[d]
        rho_d   = model.Phi_11[d, d]
        sig_inn = np.sqrt(max(1e-14, model.Sigma_ss[d, d]))
        sig_y   = sig_inn / np.sqrt(max(1e-14, 1.0 - rho_d**2))
        cover   = (g.max() - model.z_bar_state[d]) / sig_y if sig_y > 0 else 0.0
        flag(f"State grid coverage: {name}",
             cover >= 2.5,
             f"Â±{cover:.2f}Ïƒ  (recommend â‰¥ 2.5Ïƒ;  use larger state_grid_sizes for more)")

    print()
    print("=" * W)
    print("  Diagnostic report complete.")
    print("=" * W)


In [ ]:
# =============================================================================
# PART 1 EXECUTION  --  produces `model` and `pc` for the Part 2 solver
# =============================================================================
# Adjust state_grid_sizes here; everything else is config-driven.

from pathlib import Path

# Resolve dataset path robustly regardless of kernel launch directory.
_path_candidates = [
    Path("var_dataset.csv"),
    Path("TIPS_MODEL/var_dataset.csv"),
    Path("../TIPS_MODEL/var_dataset.csv"),
]
_VAR_CSV = next((p for p in _path_candidates if p.exists()), None)
if _VAR_CSV is None:
    raise FileNotFoundError(
        "Could not find var_dataset.csv. Tried: "
        + ", ".join(str(p) for p in _path_candidates)
    )
_VAR_CSV = str(_VAR_CSV)

# --- Step 1: Estimate quarterly VAR from data, then annualise ---
# The DP model uses ANNUAL periods (ages 25-80, beta=0.96).
# The VAR is estimated at quarterly frequency (T=183) for statistical power,
# then compounded to annual frequency before passing to build_model().
var_config_q, var_res, var_data = build_nominal_system1_var_config(
    csv_path=_VAR_CSV,
)
var_config = annualize_var_config(var_config_q, h=4)   # quarterly -> annual

# --- Step 2: Base (non-VAR) calibration ---
base_config = build_base_config_legacy_defaults()

# --- Step 3: Build model (economic parameters only, no grid choices here) ---
model = build_model(base_config, var_config, verbose=True)

# --- Step 4: Precompute  (numerical parameters: grid sizes, tolerances) ---
# Start with [5,5,5] for a quick sanity run; switch to [7,7,7] for production.
# If the consistency check WARNS, read ||Phi_11_off|| and ||M @ Phi_11_off||
# in the output -- those tell you whether the error is economically meaningful.
# Class defaults (warn=2e-2, error=1e-1) are used; the structural
# independence-Rouwenhorst error depends on ||M @ Phi_11_off||, not grid size.
disc_config = DiscretizationConfig(
    state_grid_sizes=(5, 5, 5),    # 125 states -- fast first run
    n_z=5,                         # reduced from 11 for speed (restore to 11 for production)
    n_eps_nodes=5,
)
pc = Precompute(model, disc_config=disc_config)
)

# --- Step 5: Full diagnostic report ---
# periods_per_year=1 because model now uses ANNUAL parameters.
# y_nom and dp are levels (not rates) so their annualised columns are N/A.
print_model_diagnostic_report(model, pc, periods_per_year=1)

# --- Part 1 complete ---
# Objects available for Part 2:  model, pc
print("\nPart 1 complete. Ready for Part 2 solver.")
print(f"  model.n_state = {model.n_state},  model.n_ret = {model.n_ret}")
print(f"  pc.N_state    = {pc.N_state},  pc.n_z = {pc.n_z},  pc.n_age = {pc.n_age}")

# LIFECYCLE PORTFOLIO CHOICE MODEL -- PART 2

## Backward Induction Solver (EGM + 2D Newton)

Requires `model` and `pc` from Part 1.

### Key adaptations from the reference EGM solver:

| Feature | Reference model | This model |
|---------|----------------|------------|
| Bequest | `theta * (W + kappa)^{-gamma}` | `b_bar * (W/A)^{-gamma} / A` (Catherine 2025) |
| Annuity factor | â€” | `A = annuity_factor(y_nom * 4, b_bar)` â€” state-dependent |
| Returns | `Rx_stock_all[j_s]` from flat table | `exp(mu_r[i_s, j_s, 0])` â€” depends on both states |
| Bill rate | `asset_ret_table[i_s, 0]` | `exp(r_bill_grid[i_s])` |
| Terminal c | Closed-form `(W+kappa)/(1 + theta^{1/gamma})` | Closed-form `W * K^{-1/gamma} / (1 + K^{-1/gamma})` |

### EGM loop (per savings point `a` on `s_grid`):
1. Solve 2D Newton for `(alpha_stock, alpha_bond)` â†’ get `euler_sum`
2. Invert Euler: `c_opt = (beta * euler_sum)^{-1/gamma}`
3. Endogenous wealth: `x_endo = c_opt + a`
4. Interpolate policy onto exogenous `wealth_grid`

In [23]:
"""
LIFECYCLE PORTFOLIO CHOICE MODEL - PART 2
Backward Induction Solver with Endogenous Grid Method (EGM)

Three assets: Bills, Stocks, Nominal Bonds
Catherine (2025) bequest motive: b(W, A) = b_bar * (W/A)^(1-gamma) / (1-gamma)
  where A = annuity_factor(y_nom * 4, b_bar) is state-dependent.

Returns architecture: mu_r[i_s, j_s, :] gives E[xr, xb | s_t=i_s, s_{t+1}=j_s].
  Rx_stock[j_s] = exp(mu_r[i_s, j_s, 0]),  Rx_bond[j_s] = exp(mu_r[i_s, j_s, 1])
  Precomputed per current state i_s before Newton iterations.

Policy functions output:
    C_mat[t, i_z, i_s, i_w]  -- optimal consumption
    S_mat[t, i_z, i_s, i_w]  -- optimal stock share  (alpha_stock)
    B_mat[t, i_z, i_s, i_w]  -- optimal bond share   (alpha_bond)
"""

import numpy as np
from numba import njit, prange
from math import exp
import time

# =============================================================================
# DIAGNOSTIC CONSTANTS
# =============================================================================
# Integer counter indices -- diag_int[i_s, idx], shape (N_state, N_DIAG_INT)
DI_CORNER_BILLS    = 0   # all-bills corner solution
DI_CORNER_STOCKS   = 1   # all-stocks corner
DI_CORNER_BONDS    = 2   # all-bonds corner
DI_EDGE_SB         = 3   # stock + bill edge
DI_EDGE_BB         = 4   # bond + bill edge
DI_EDGE_STOCKBOND  = 5   # stock + bond edge (no bills)
DI_INTERIOR        = 6   # interior Newton converged
DI_NEWTON_FAIL     = 7   # Newton hit max_iter without converging
DI_SINGULAR_JAC    = 8   # singular Jacobian event
DI_TINY_SAVINGS    = 9   # s_val < threshold, trivial all-bills
DI_TOTAL_CALLS     = 10  # total Newton calls
DI_NEG_CONSUMPTION = 11  # negative euler (before clamping)
DI_MONO_VIOLATIONS = 12  # EGM monotonicity violations
N_DIAG_INT = 13

# Float counter indices -- diag_float[i_s, idx], shape (N_state, N_DIAG_FLOAT)
DF_WORST_MONO_DROP  = 0  # largest endogenous grid inversion
DF_MAX_FOC_RESID    = 1  # worst FOC residual at Newton exit
DF_SUM_FOC_RESID_SQ = 2  # sum of squared FOC residuals (for RMS)
DF_MIN_ALPHA_S      = 3  # min stock share
DF_MAX_ALPHA_S      = 4  # max stock share
DF_MIN_ALPHA_B      = 5  # min bond share
DF_MAX_ALPHA_B      = 6  # max bond share
DF_SUM_ALPHA_S      = 7  # sum of stock shares (for mean)
DF_SUM_ALPHA_B      = 8  # sum of bond shares (for mean)
N_DIAG_FLOAT = 9

# Exit codes from Newton solvers
EC_TINY_SAVINGS = 0
EC_CORNER_BILLS = 1
EC_CORNER_STOCKS = 2
EC_CORNER_BONDS = 3
EC_EDGE_SB = 4       # stock + bill
EC_EDGE_BB = 5       # bond + bill
EC_EDGE_STOCKBOND = 6 # stock + bond
EC_INTERIOR = 7       # interior Newton converged
EC_NEWTON_FAIL = 8    # Newton did not converge

# Mapping from exit_code to diag_int index
_EC_TO_DI = np.array([
    DI_TINY_SAVINGS,    # EC=0
    DI_CORNER_BILLS,    # EC=1
    DI_CORNER_STOCKS,   # EC=2
    DI_CORNER_BONDS,    # EC=3
    DI_EDGE_SB,         # EC=4
    DI_EDGE_BB,         # EC=5
    DI_EDGE_STOCKBOND,  # EC=6
    DI_INTERIOR,        # EC=7
    DI_NEWTON_FAIL,     # EC=8
], dtype=np.int64)


# =============================================================================
# HELPERS: INTERPOLATION AND SIMPLEX PROJECTION
# =============================================================================

@njit(fastmath=True)
def fast_interp_slope_1d(x, x_grid, y_grid):
    """
    Slope of the piecewise-linear interpolant -- i.e. dc_next/dx_next (MPC).
    Same binary search as fast_interp_1d, but returns the interval slope.
    At the extrapolation boundaries, returns the nearest interior slope.
    """
    n = len(x_grid)
    if n < 2:
        return 0.0
    if x <= x_grid[0]:
        return (y_grid[1] - y_grid[0]) / (x_grid[1] - x_grid[0] + 1e-30)
    if x >= x_grid[n - 1]:
        return (y_grid[n-1] - y_grid[n-2]) / (x_grid[n-1] - x_grid[n-2] + 1e-30)
    lo, hi = 0, n - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if x_grid[mid] <= x:
            lo = mid
        else:
            hi = mid
    dx = x_grid[hi] - x_grid[lo]
    if dx < 1e-30:
        return 0.0
    return (y_grid[hi] - y_grid[lo]) / dx


@njit(fastmath=True)
def fast_interp_1d(x, x_grid, y_grid):
    """Linear interpolation on a sorted grid with binary search.
    Uses linear extrapolation beyond grid boundaries."""
    n = len(x_grid)
    if x <= x_grid[0]:
        dx = x_grid[1] - x_grid[0] + 1e-30
        return y_grid[0] + (y_grid[1] - y_grid[0]) * (x - x_grid[0]) / dx
    if x >= x_grid[n - 1]:
        dx = x_grid[n - 1] - x_grid[n - 2] + 1e-30
        return y_grid[n - 1] + (y_grid[n - 1] - y_grid[n - 2]) * (x - x_grid[n - 1]) / dx
    lo, hi = 0, n - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if x_grid[mid] <= x:
            lo = mid
        else:
            hi = mid
    x0, x1 = x_grid[lo], x_grid[hi]
    y0, y1 = y_grid[lo], y_grid[hi]
    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)


@njit(fastmath=True)
def fast_interp_1d_with_slope(x, x_grid, y_grid):
    """Combined interpolation: returns (value, slope) from a single binary search."""
    n = len(x_grid)
    if n < 2:
        return y_grid[0], 0.0
    if x <= x_grid[0]:
        dx = x_grid[1] - x_grid[0] + 1e-30
        slope = (y_grid[1] - y_grid[0]) / dx
        val = y_grid[0] + slope * (x - x_grid[0])
        return val, slope
    if x >= x_grid[n - 1]:
        dx = x_grid[n - 1] - x_grid[n - 2] + 1e-30
        slope = (y_grid[n-1] - y_grid[n-2]) / dx
        val = y_grid[n - 1] + slope * (x - x_grid[n - 1])
        return val, slope
    lo, hi = 0, n - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if x_grid[mid] <= x:
            lo = mid
        else:
            hi = mid
    x0, x1 = x_grid[lo], x_grid[hi]
    y0, y1 = y_grid[lo], y_grid[hi]
    dx = x1 - x0
    if dx < 1e-30:
        return y0, 0.0
    slope = (y1 - y0) / dx
    val = y0 + slope * (x - x0)
    return val, slope


@njit(fastmath=True)
def project_to_triangle(alpha_s, alpha_b):
    """Project (alpha_s, alpha_b) onto feasible region: >=0, sum<=1."""
    alpha_s = max(0.0, alpha_s)
    alpha_b = max(0.0, alpha_b)
    if alpha_s + alpha_b > 1.0:
        excess = (alpha_s + alpha_b - 1.0) * 0.5
        alpha_s = max(0.0, min(1.0, alpha_s - excess))
        alpha_b = max(0.0, min(1.0, alpha_b - excess))
    return alpha_s, alpha_b


# =============================================================================
# FOC AND JACOBIAN -- RETIREMENT
# =============================================================================


@njit(fastmath=True)
def compute_foc_jac_retirement(alpha_s, alpha_b, s_val, z_idx, i_s,
                                wealth_grid, c_next_full, pension_next_scalar,
                                annuity_factor_is,
                                Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                                gamma, psi, beta, b_bar,
                             min_wealth_inv=1e-10, min_consumption=1e-10,
                             prob_skip=1e-12):
    a_bill     = 1.0 - alpha_s - alpha_b
    prob_death = 1.0 - psi

    foc_s = 0.0; foc_b = 0.0
    J_ss  = 0.0; J_bb  = 0.0; J_sb  = 0.0
    euler_sum = 0.0

    N_state = Pi_state.shape[1]

    for j_s in range(N_state):
        pi_s = Pi_state[i_s, j_s]
        if pi_s < prob_skip:
            continue

        R_s = R_bill * Rx_stock_next[j_s]
        R_b = R_bill * Rx_bond_next[j_s]
        R_p = alpha_s * R_s + alpha_b * R_b + a_bill * R_bill

        Rex_s = R_s - R_bill
        Rex_b = R_b - R_bill

        w_inv  = max(s_val * R_p, min_wealth_inv)
        x_next = w_inv + pension_next_scalar

        c_row  = c_next_full[j_s, :]
        c_next, mpc = fast_interp_1d_with_slope(x_next, wealth_grid, c_row)
        c_next = max(c_next, min_consumption)
        mpc = max(0.0, min(1.0, mpc))

        mu_alive  = c_next ** (-gamma)
        w_A        = w_inv / annuity_factor_is
        mu_bequest = b_bar * w_A ** (-gamma) / annuity_factor_is
        mu_comb    = psi * mu_alive + prob_death * mu_bequest

        mup_alive   = -gamma * mu_alive / c_next * mpc
        mup_bequest = -gamma * mu_bequest / (w_A * annuity_factor_is)
        mup_comb    = psi * mup_alive + prob_death * mup_bequest

        wmu  = pi_s * mu_comb
        wmup = pi_s * mup_comb

        euler_sum += wmu * R_p
        foc_s     += wmu * Rex_s
        foc_b     += wmu * Rex_b

        jac   = wmup * s_val
        J_ss += jac * Rex_s * Rex_s
        J_bb += jac * Rex_b * Rex_b
        J_sb += jac * Rex_s * Rex_b

    return foc_s, foc_b, J_ss, J_bb, J_sb, euler_sum


# =============================================================================
# NEWTON PORTFOLIO SOLVER -- RETIREMENT
# =============================================================================

@njit(fastmath=True)
def solve_portfolio_2d_retirement(s_val, z_idx, i_s,
                                   wealth_grid, c_next_full, pension_next_scalar,
                                   annuity_factor_is,
                                   Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                                   gamma, psi, beta, b_bar,
                                   init_s=0.1, init_b=0.4,
                                   tol=1e-7, max_iter=20,
                                   tiny_savings=1e-6, corner_tol=1e-8,
                                   edge_max_iter=8, edge_accept_factor=10.0,
                                   singular_det=1e-15, grad_step_size=0.05,
                                   step_damp=0.2, grad_denom_eps=1e-10,
                                   min_wealth_inv=1e-10, min_consumption=1e-10,
                                   prob_skip=1e-12):
    """2D Newton-Raphson for optimal (alpha_stock, alpha_bond) in retirement.
    Returns: (alpha_s, alpha_b, euler_sum, exit_code, foc_resid)"""

    # Tiny savings: all bills
    if s_val < tiny_savings:
        _, _, _, _, _, e = compute_foc_jac_retirement(
            0.0, 0.0, s_val, z_idx, i_s,
            wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        return 0.0, 0.0, e, EC_TINY_SAVINGS, 0.0

    # Corner: all bills
    fs0, fb0, _, _, _, e0 = compute_foc_jac_retirement(
        0.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    scale = max(abs(e0), 1.0)  # FOC scale for relative tolerance
    if fs0 <= corner_tol * scale and fb0 <= corner_tol * scale:
        return 0.0, 0.0, e0, EC_CORNER_BILLS, 0.0

    # Corner: all stocks
    fs1, fb1, _, _, _, e1 = compute_foc_jac_retirement(
        1.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    if fs1 >= -corner_tol * scale and fb1 <= fs1 + corner_tol * scale:
        return 1.0, 0.0, e1, EC_CORNER_STOCKS, 0.0

    # Corner: all bonds
    fs2, fb2, _, _, _, e2 = compute_foc_jac_retirement(
        0.0, 1.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    if fb2 >= -corner_tol * scale and fs2 <= fb2 + corner_tol * scale:
        return 0.0, 1.0, e2, EC_CORNER_BONDS, 0.0

    # Edge: stocks + bills only (alpha_b = 0)
    if fs0 > 0.0 and fs1 < 0.0:
        a_s = fs0 / (fs0 - fs1)
        fs = fs0  # init for residual tracking
        for _ in range(edge_max_iter):
            fs, fb, Jss, _, _, e = compute_foc_jac_retirement(
                a_s, 0.0, s_val, z_idx, i_s,
                wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
                Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            if abs(fs) < tol * scale:
                break
            if abs(Jss) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - fs / Jss))
        if abs(fs) < tol * scale * edge_accept_factor and fb <= tol * scale:
            return a_s, 0.0, e, EC_EDGE_SB, abs(fs) / scale

    # Edge: bonds + bills only (alpha_s = 0)
    if fb0 > 0.0 and fb2 < 0.0:
        a_b = fb0 / (fb0 - fb2)
        fb = fb0
        for _ in range(edge_max_iter):
            fs, fb, _, Jbb, _, e = compute_foc_jac_retirement(
                0.0, a_b, s_val, z_idx, i_s,
                wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
                Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            if abs(fb) < tol * scale:
                break
            if abs(Jbb) < singular_det:
                break
            a_b = max(0.0, min(1.0, a_b - fb / Jbb))
        if abs(fb) < tol * scale * edge_accept_factor and fs <= tol * scale:
            return 0.0, a_b, e, EC_EDGE_BB, abs(fb) / scale

    # Edge: stocks + bonds only (no bills, alpha_s + alpha_b = 1)
    g1 = fs1 - fb1
    g2 = fs2 - fb2
    if g1 * g2 < 0.0:
        a_s = g2 / (g2 - g1)
        g = g2  # init
        for _ in range(edge_max_iter):
            a_b = 1.0 - a_s
            fs, fb, Jss, Jbb, Jsb, e = compute_foc_jac_retirement(
                a_s, a_b, s_val, z_idx, i_s,
                wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
                Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            g = fs - fb
            if abs(g) < tol * scale:
                break
            dg = Jss - 2.0 * Jsb + Jbb
            if abs(dg) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - g / dg))
        if abs(fs - fb) < tol * scale * edge_accept_factor and fs >= -tol * scale:
            return a_s, 1.0 - a_s, e, EC_EDGE_STOCKBOND, abs(g) / scale

    # Interior Newton-Raphson
    a_s = init_s
    a_b = init_b
    e_last = 0.0
    err = 1.0

    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb, e_sum = compute_foc_jac_retirement(
            a_s, a_b, s_val, z_idx, i_s,
            wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        e_last = e_sum

        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, e_last, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s, a_b = project_to_triangle(a_s + step_s, a_b + step_b)

    return a_s, a_b, e_last, EC_NEWTON_FAIL, err / scale

# =============================================================================
# UNCONSTRAINED PORTFOLIO SOLVER -- RETIREMENT
# =============================================================================

@njit(fastmath=True)
def solve_portfolio_unconstrained_retirement(s_val, z_idx, i_s,
                                              wealth_grid, c_next_full, pension_next_scalar,
                                              annuity_factor_is,
                                              Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                                              gamma, psi, beta, b_bar,
                                              init_s=0.1, init_b=0.4,
                                              tol=1e-7, max_iter=30,
                                              tiny_savings=1e-6,
                                              singular_det=1e-15, grad_step_size=0.05,
                                              step_damp=0.3, grad_denom_eps=1e-10,
                                              min_wealth_inv=1e-10, min_consumption=1e-10,
                                   prob_skip=1e-12):
    """Unconstrained Newton for (alpha_stock, alpha_bond) in retirement.
    No short-sale or leverage constraints."""

    if s_val < tiny_savings:
        _, _, _, _, _, e = compute_foc_jac_retirement(
            0.0, 0.0, s_val, z_idx, i_s,
            wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        return 0.0, 0.0, e, EC_TINY_SAVINGS, 0.0

    # Scale from all-bills FOC
    _, _, _, _, _, e0 = compute_foc_jac_retirement(
        0.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    scale = max(abs(e0), 1.0)

    a_s = init_s
    a_b = init_b
    e_last = 0.0
    err = 1.0

    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb, e_sum = compute_foc_jac_retirement(
            a_s, a_b, s_val, z_idx, i_s,
            wealth_grid, c_next_full, pension_next_scalar, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        e_last = e_sum

        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, e_last, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s += step_s
        a_b += step_b

    return a_s, a_b, e_last, EC_NEWTON_FAIL, err / scale


# =============================================================================
# FOC AND JACOBIAN -- WORKING AGE
# =============================================================================

@njit(fastmath=True)
def compute_foc_jac_working(alpha_s, alpha_b, s_val, z_idx, i_s,
                             wealth_grid, c_next_full, income_next_table,
                             annuity_factor_is,
                             Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                             eps_nodes, eps_weights,
                             gamma, psi, beta, b_bar,
                             min_wealth_inv=1e-10, min_consumption=1e-10,
                             prob_skip=1e-12):
    a_bill     = 1.0 - alpha_s - alpha_b
    prob_death = 1.0 - psi

    foc_s = 0.0; foc_b = 0.0
    J_ss  = 0.0; J_bb  = 0.0; J_sb  = 0.0
    euler_sum = 0.0

    N_state = Pi_state.shape[1]
    n_z     = Pi_z.shape[1]
    n_eps   = len(eps_nodes)

    for j_s in range(N_state):
        p_var = Pi_state[i_s, j_s]
        if p_var < prob_skip:
            continue

        R_s = R_bill * Rx_stock_next[j_s]
        R_b = R_bill * Rx_bond_next[j_s]
        R_p = alpha_s * R_s + alpha_b * R_b + a_bill * R_bill

        Rex_s = R_s - R_bill
        Rex_b = R_b - R_bill

        w_inv = max(s_val * R_p, min_wealth_inv)

        w_A         = w_inv / annuity_factor_is
        mu_bequest  = b_bar * w_A ** (-gamma) / annuity_factor_is
        mup_bequest = -gamma * mu_bequest / (w_A * annuity_factor_is)

        # -- bequest contribution: once per j_s (independent of income) --
        death_mu  = p_var * prob_death * mu_bequest
        death_mup = p_var * prob_death * mup_bequest

        euler_sum += death_mu * R_p
        foc_s     += death_mu * Rex_s
        foc_b     += death_mu * Rex_b

        jac_b  = death_mup * s_val
        J_ss  += jac_b * Rex_s * Rex_s
        J_bb  += jac_b * Rex_b * Rex_b
        J_sb  += jac_b * Rex_s * Rex_b

        # -- alive contribution: inner loops over income uncertainty --
        for j_z in range(n_z):
            p_z = Pi_z[z_idx, j_z]
            if p_z < prob_skip:
                continue

            p_out = p_var * p_z

            for i_e in range(n_eps):
                weight = p_out * eps_weights[i_e]

                income_next = income_next_table[j_z, i_e]
                x_next = w_inv + income_next

                c_row  = c_next_full[j_z, j_s, :]
                c_next, mpc = fast_interp_1d_with_slope(x_next, wealth_grid, c_row)
                c_next = max(c_next, min_consumption)
                mpc = max(0.0, min(1.0, mpc))

                mu_alive  = c_next ** (-gamma)
                mup_alive = -gamma * mu_alive / c_next * mpc

                wmu  = weight * psi * mu_alive
                wmup = weight * psi * mup_alive

                euler_sum += wmu * R_p
                foc_s     += wmu * Rex_s
                foc_b     += wmu * Rex_b

                jac = wmup * s_val
                J_ss += jac * Rex_s * Rex_s
                J_bb += jac * Rex_b * Rex_b
                J_sb += jac * Rex_s * Rex_b

    return foc_s, foc_b, J_ss, J_bb, J_sb, euler_sum

# =============================================================================
# NEWTON PORTFOLIO SOLVER -- WORKING AGE
# =============================================================================

@njit(fastmath=True)
def solve_portfolio_2d_working(s_val, z_idx, i_s,
                                wealth_grid, c_next_full, income_next_table,
                                annuity_factor_is,
                                Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                                eps_nodes, eps_weights,
                                gamma, psi, beta, b_bar,
                                init_s=0.1, init_b=0.4,
                                tol=1e-7, max_iter=20,
                                   tiny_savings=1e-6, corner_tol=1e-8,
                                   edge_max_iter=8, edge_accept_factor=10.0,
                                   singular_det=1e-15, grad_step_size=0.05,
                                   step_damp=0.2, grad_denom_eps=1e-10,
                                   min_wealth_inv=1e-10, min_consumption=1e-10,
                                   prob_skip=1e-12):
    """2D Newton-Raphson for optimal (alpha_stock, alpha_bond) during working years.
    Returns: (alpha_s, alpha_b, euler_sum, exit_code, foc_resid)"""

    if s_val < tiny_savings:
        _, _, _, _, _, e = compute_foc_jac_working(
            0.0, 0.0, s_val, z_idx, i_s,
            wealth_grid, c_next_full, income_next_table, annuity_factor_is,
            Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
            eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        return 0.0, 0.0, e, EC_TINY_SAVINGS, 0.0

    # Corner: all bills
    fs0, fb0, _, _, _, e0 = compute_foc_jac_working(
        0.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, income_next_table, annuity_factor_is,
        Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
        eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    scale = max(abs(e0), 1.0)  # FOC scale for relative tolerance
    if fs0 <= corner_tol * scale and fb0 <= corner_tol * scale:
        return 0.0, 0.0, e0, EC_CORNER_BILLS, 0.0

    # Corner: all stocks
    fs1, fb1, _, _, _, e1 = compute_foc_jac_working(
        1.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, income_next_table, annuity_factor_is,
        Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
        eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    if fs1 >= -corner_tol * scale and fb1 <= fs1 + corner_tol * scale:
        return 1.0, 0.0, e1, EC_CORNER_STOCKS, 0.0

    # Corner: all bonds
    fs2, fb2, _, _, _, e2 = compute_foc_jac_working(
        0.0, 1.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, income_next_table, annuity_factor_is,
        Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
        eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    if fb2 >= -corner_tol * scale and fs2 <= fb2 + corner_tol * scale:
        return 0.0, 1.0, e2, EC_CORNER_BONDS, 0.0

    # Edge: stocks + bills only (alpha_b = 0)
    if fs0 > 0.0 and fs1 < 0.0:
        a_s = fs0 / (fs0 - fs1)
        fs = fs0
        for _ in range(edge_max_iter):
            fs, fb, Jss, _, _, e = compute_foc_jac_working(
                a_s, 0.0, s_val, z_idx, i_s,
                wealth_grid, c_next_full, income_next_table, annuity_factor_is,
                Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            if abs(fs) < tol * scale:
                break
            if abs(Jss) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - fs / Jss))
        if abs(fs) < tol * scale * edge_accept_factor and fb <= tol * scale:
            return a_s, 0.0, e, EC_EDGE_SB, abs(fs) / scale

    # Edge: bonds + bills only (alpha_s = 0)
    if fb0 > 0.0 and fb2 < 0.0:
        a_b = fb0 / (fb0 - fb2)
        fb = fb0
        for _ in range(edge_max_iter):
            fs, fb, _, Jbb, _, e = compute_foc_jac_working(
                0.0, a_b, s_val, z_idx, i_s,
                wealth_grid, c_next_full, income_next_table, annuity_factor_is,
                Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            if abs(fb) < tol * scale:
                break
            if abs(Jbb) < singular_det:
                break
            a_b = max(0.0, min(1.0, a_b - fb / Jbb))
        if abs(fb) < tol * scale * edge_accept_factor and fs <= tol * scale:
            return 0.0, a_b, e, EC_EDGE_BB, abs(fb) / scale

    # Edge: stocks + bonds only (no bills, alpha_s + alpha_b = 1)
    g1 = fs1 - fb1
    g2 = fs2 - fb2
    if g1 * g2 < 0.0:
        a_s = g2 / (g2 - g1)
        g = g2
        for _ in range(edge_max_iter):
            a_b = 1.0 - a_s
            fs, fb, Jss, Jbb, Jsb, e = compute_foc_jac_working(
                a_s, a_b, s_val, z_idx, i_s,
                wealth_grid, c_next_full, income_next_table, annuity_factor_is,
                Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
            g = fs - fb
            if abs(g) < tol * scale:
                break
            dg = Jss - 2.0 * Jsb + Jbb
            if abs(dg) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - g / dg))
        if abs(fs - fb) < tol * scale * edge_accept_factor and fs >= -tol * scale:
            return a_s, 1.0 - a_s, e, EC_EDGE_STOCKBOND, abs(g) / scale

    # Interior Newton-Raphson
    a_s = init_s
    a_b = init_b
    e_last = 0.0
    err = 1.0

    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb, e_sum = compute_foc_jac_working(
            a_s, a_b, s_val, z_idx, i_s,
            wealth_grid, c_next_full, income_next_table, annuity_factor_is,
            Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
            eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        e_last = e_sum

        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, e_last, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s, a_b = project_to_triangle(a_s + step_s, a_b + step_b)

    return a_s, a_b, e_last, EC_NEWTON_FAIL, err / scale


# =============================================================================
# UNCONSTRAINED PORTFOLIO SOLVER -- TERMINAL
# =============================================================================

@njit(fastmath=True)
def solve_portfolio_unconstrained_terminal(i_s, Pi_state, Rx_stock_next, Rx_bond_next,
                                            R_bill, gamma,
                                            init_s=0.1, init_b=0.4,
                                            tol=1e-7, max_iter=30,
                                            singular_det=1e-15, grad_step_size=0.05,
                                            step_damp=0.3, grad_denom_eps=1e-10,
                                            min_return_power=1e-15,
                                   prob_skip=1e-12):
    """Unconstrained Newton for terminal portfolio.
    No short-sale or leverage constraints.
    Returns: (alpha_s, alpha_b, exit_code, foc_resid)"""

    scale = R_bill ** (-gamma)

    a_s = init_s
    a_b = init_b
    err = 1.0

    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb = compute_terminal_portfolio_foc_jac(
            a_s, a_b, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)

        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s += step_s
        a_b += step_b

    return a_s, a_b, EC_NEWTON_FAIL, err / scale

# =============================================================================
# TERMINAL CONDITION
# =============================================================================

# =============================================================================
# UNCONSTRAINED PORTFOLIO SOLVER -- WORKING AGE
# =============================================================================

@njit(fastmath=True)
def solve_portfolio_unconstrained_working(s_val, z_idx, i_s,
                                           wealth_grid, c_next_full, income_next_table,
                                           annuity_factor_is,
                                           Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                                           eps_nodes, eps_weights,
                                           gamma, psi, beta, b_bar,
                                           init_s=0.1, init_b=0.4,
                                           tol=1e-7, max_iter=30,
                                              tiny_savings=1e-6,
                                              singular_det=1e-15, grad_step_size=0.05,
                                              step_damp=0.3, grad_denom_eps=1e-10,
                                              min_wealth_inv=1e-10, min_consumption=1e-10,
                                   prob_skip=1e-12):
    """Unconstrained Newton for (alpha_stock, alpha_bond) at working age.
    No short-sale or leverage constraints."""

    if s_val < tiny_savings:
        _, _, _, _, _, e = compute_foc_jac_working(
            0.0, 0.0, s_val, z_idx, i_s,
            wealth_grid, c_next_full, income_next_table, annuity_factor_is,
            Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
            eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        return 0.0, 0.0, e, EC_TINY_SAVINGS, 0.0

    # Scale from all-bills FOC
    _, _, _, _, _, e0 = compute_foc_jac_working(
        0.0, 0.0, s_val, z_idx, i_s,
        wealth_grid, c_next_full, income_next_table, annuity_factor_is,
        Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
        eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
    scale = max(abs(e0), 1.0)

    a_s = init_s
    a_b = init_b
    e_last = 0.0
    err = 1.0

    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb, e_sum = compute_foc_jac_working(
            a_s, a_b, s_val, z_idx, i_s,
            wealth_grid, c_next_full, income_next_table, annuity_factor_is,
            Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
            eps_nodes, eps_weights, gamma, psi, beta, b_bar,
            min_wealth_inv, min_consumption, prob_skip)
        e_last = e_sum

        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, e_last, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s += step_s
        a_b += step_b

    return a_s, a_b, e_last, EC_NEWTON_FAIL, err / scale


# =============================================================================
# TERMINAL AGE PORTFOLIO SOLVER
# =============================================================================

@njit(fastmath=True)
def compute_terminal_portfolio_foc_jac(alpha_s, alpha_b, i_s,
                                        Pi_state, Rx_stock_next, Rx_bond_next,
                                        R_bill, gamma,
                                          min_return_power=1e-15,
                                          prob_skip=1e-12):
    """
    FOC and Jacobian for the terminal-period portfolio problem.

    Because the bequest b(a*R_port, A) = b_bar*(a*R_port/A)^{1-gamma}/(1-gamma)
    is CRRA in terminal wealth, the portfolio FOC is proportional to a^{1-gamma}
    and therefore independent of savings a (hence independent of W and c).

    FOC_k = sum_j pi(j|i) * R_port(j)^{-gamma} * (R_k(j) - R_bill) = 0,  k in {s,b}
    J_kl  = sum_j pi(j|i) * (-gamma) * R_port(j)^{-gamma-1} * Rex_k * Rex_l
    """
    foc_s = 0.0;  foc_b = 0.0
    J_ss  = 0.0;  J_bb  = 0.0;  J_sb  = 0.0
    a_bill = 1.0 - alpha_s - alpha_b

    N_state = Pi_state.shape[1]
    for j_s in range(N_state):
        pi_s = Pi_state[i_s, j_s]
        if pi_s < prob_skip:
            continue
        R_s   = R_bill * Rx_stock_next[j_s]
        R_b   = R_bill * Rx_bond_next[j_s]
        R_p   = alpha_s * R_s + alpha_b * R_b + a_bill * R_bill
        Rex_s = R_s - R_bill
        Rex_b = R_b - R_bill

        Rp_mg  = max(R_p, min_return_power) ** (-gamma)
        Rp_mg1 = max(R_p, min_return_power) ** (-gamma - 1.0)

        foc_s += pi_s * Rp_mg * Rex_s
        foc_b += pi_s * Rp_mg * Rex_b

        jac   = pi_s * (-gamma) * Rp_mg1
        J_ss += jac * Rex_s * Rex_s
        J_bb += jac * Rex_b * Rex_b
        J_sb += jac * Rex_s * Rex_b

    return foc_s, foc_b, J_ss, J_bb, J_sb


@njit(fastmath=True)
def solve_portfolio_2d_terminal(i_s, Pi_state, Rx_stock_next, Rx_bond_next,
                                 R_bill, gamma,
                                 init_s=0.1, init_b=0.4, tol=1e-7, max_iter=20,
                                    tiny_savings=1e-6, corner_tol=1e-8,
                                    edge_max_iter=8, edge_accept_factor=10.0,
                                    singular_det=1e-15, grad_step_size=0.05,
                                    step_damp=0.2, grad_denom_eps=1e-10,
                                    min_return_power=1e-15,
                                   prob_skip=1e-12):
    """2D Newton-Raphson for optimal (alpha_s, alpha_b) at the terminal age.
    Returns: (alpha_s, alpha_b, exit_code, foc_resid)"""

    scale = R_bill ** (-gamma)  # FOC scale for relative tolerance

    # Corner: all bills
    fs0, fb0, _, _, _ = compute_terminal_portfolio_foc_jac(
        0.0, 0.0, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
    if fs0 <= corner_tol * scale and fb0 <= corner_tol * scale:
        return 0.0, 0.0, EC_CORNER_BILLS, 0.0

    # Corner: all stocks
    fs1, fb1, _, _, _ = compute_terminal_portfolio_foc_jac(
        1.0, 0.0, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
    if fs1 >= -corner_tol * scale and fb1 <= fs1 + corner_tol * scale:
        return 1.0, 0.0, EC_CORNER_STOCKS, 0.0

    # Corner: all bonds
    fs2, fb2, _, _, _ = compute_terminal_portfolio_foc_jac(
        0.0, 1.0, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
    if fb2 >= -corner_tol * scale and fs2 <= fb2 + corner_tol * scale:
        return 0.0, 1.0, EC_CORNER_BONDS, 0.0

    # Edge: stocks + bills only (alpha_b = 0)
    if fs0 > 0.0 and fs1 < 0.0:
        a_s = fs0 / (fs0 - fs1)
        fs = fs0
        for _ in range(edge_max_iter):
            fs, fb, Jss, _, _ = compute_terminal_portfolio_foc_jac(
                a_s, 0.0, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
            if abs(fs) < tol * scale:
                break
            if abs(Jss) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - fs / Jss))
        if abs(fs) < tol * scale * edge_accept_factor and fb <= tol * scale:
            return a_s, 0.0, EC_EDGE_SB, abs(fs) / scale

    # Edge: bonds + bills only (alpha_s = 0)
    if fb0 > 0.0 and fb2 < 0.0:
        a_b = fb0 / (fb0 - fb2)
        fb = fb0
        for _ in range(edge_max_iter):
            fs, fb, _, Jbb, _ = compute_terminal_portfolio_foc_jac(
                0.0, a_b, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
            if abs(fb) < tol * scale:
                break
            if abs(Jbb) < singular_det:
                break
            a_b = max(0.0, min(1.0, a_b - fb / Jbb))
        if abs(fb) < tol * scale * edge_accept_factor and fs <= tol * scale:
            return 0.0, a_b, EC_EDGE_BB, abs(fb) / scale

    # Edge: stocks + bonds only (no bills, alpha_s + alpha_b = 1)
    g1 = fs1 - fb1
    g2 = fs2 - fb2
    if g1 * g2 < 0.0:
        a_s = g2 / (g2 - g1)
        g = g2
        for _ in range(edge_max_iter):
            a_b = 1.0 - a_s
            fs, fb, Jss, Jbb, Jsb = compute_terminal_portfolio_foc_jac(
                a_s, a_b, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
            g = fs - fb
            if abs(g) < tol * scale:
                break
            dg = Jss - 2.0 * Jsb + Jbb
            if abs(dg) < singular_det:
                break
            a_s = max(0.0, min(1.0, a_s - g / dg))
        if abs(fs - fb) < tol * scale * edge_accept_factor and fs >= -tol * scale:
            return a_s, 1.0 - a_s, EC_EDGE_STOCKBOND, abs(g) / scale

    # Interior Newton-Raphson
    a_s = init_s
    a_b = init_b
    err = 1.0
    for _ in range(max_iter):
        fs, fb, Jss, Jbb, Jsb = compute_terminal_portfolio_foc_jac(
            a_s, a_b, i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, min_return_power, prob_skip)
        err = (fs * fs + fb * fb) ** 0.5
        if err < tol * scale:
            return a_s, a_b, EC_INTERIOR, err / scale

        det = Jss * Jbb - Jsb * Jsb
        if abs(det) < singular_det:
            step_s = grad_step_size * fs / (err + grad_denom_eps)
            step_b = grad_step_size * fb / (err + grad_denom_eps)
        else:
            inv_d  = 1.0 / det
            step_s = -(Jbb * fs - Jsb * fb) * inv_d
            step_b = -(-Jsb * fs + Jss * fb) * inv_d

        slen = (step_s * step_s + step_b * step_b) ** 0.5
        if slen > step_damp:
            sc    = step_damp / slen
            step_s *= sc
            step_b *= sc

        a_s, a_b = project_to_triangle(a_s + step_s, a_b + step_b)

    return a_s, a_b, EC_NEWTON_FAIL, err / scale


@njit
def solve_terminal_age(wealth_grid, annuity_factors, r_bill_grid, Pi_state, mu_r,
                        gamma, beta, b_bar, N_state, n_z, constrained=True, solver_config=None,
                        min_return_power=1e-15, min_consumption=1e-10):
    """
    Terminal period: jointly solve for c* and optimal (alpha_s*, alpha_b*).

    Portfolio FOC (independent of c and W due to CRRA bequest structure):
        sum_j pi(j|i) * R_port*(j)^{-gamma} * (R_k(j) - R_bill) = 0,  k in {s,b}
    Solved once per financial state i_s via solve_portfolio_2d_terminal.

    Consumption closed-form (derived after solving portfolio):
        Omega = b_bar * A^{gamma-1} * sum_j pi(j|i) * R_port*(j)^{1-gamma}
        ratio = (beta * Omega)^{-1/gamma}       -- = c*/a* at optimum
        c*    = W * ratio / (1 + ratio)

    The portfolio decouples from c because the bequest is homogeneous of
    degree 1-gamma in wealth, so the portfolio FOC is proportional to a^{1-gamma}.

    Output shapes: each (n_z, N_state, n_w)
    Also returns terminal_diag_int (N_state,) with exit codes for diagnostics.
    """
    if solver_config is None:
        solver_config = SolverConfig()
    sc = solver_config

    n_w = len(wealth_grid)
    out_c       = np.empty((n_z, N_state, n_w))
    out_alpha_s = np.empty((n_z, N_state, n_w))
    out_alpha_b = np.empty((n_z, N_state, n_w))
    terminal_diag_int = np.zeros(N_state, dtype=np.int64)  # exit codes

    for i_s in range(N_state):
        R_bill = exp(r_bill_grid[i_s])
        A_is   = annuity_factors[i_s]

        Rx_stock_next = np.empty(N_state)
        Rx_bond_next  = np.empty(N_state)
        for j_s in range(N_state):
            Rx_stock_next[j_s] = exp(mu_r[i_s, j_s, 0])
            Rx_bond_next[j_s]  = exp(mu_r[i_s, j_s, 1])

        # Optimal terminal portfolio -- same for all W and z at this financial state
        if constrained:
            opt_s, opt_b, exit_code, foc_resid = solve_portfolio_2d_terminal(
            i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma,
                min_return_power=min_return_power,
                tol=sc.tol, max_iter=sc.max_iter,
                corner_tol=sc.corner_tol, edge_max_iter=sc.edge_max_iter,
                edge_accept_factor=sc.edge_accept_factor,
                singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                step_damp=sc.step_damp_constrained, grad_denom_eps=sc.grad_denom_eps,
                prob_skip=sc.prob_skip_threshold)
        else:
            opt_s, opt_b, exit_code, foc_resid = solve_portfolio_unconstrained_terminal(
            i_s, Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma,
                min_return_power=min_return_power,
                tol=sc.tol, max_iter=sc.max_iter_unconstrained,
                singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                step_damp=sc.step_damp_unconstrained, grad_denom_eps=sc.grad_denom_eps,
                prob_skip=sc.prob_skip_threshold)
        terminal_diag_int[i_s] = exit_code
        a_bill = 1.0 - opt_s - opt_b

        # Omega = b_bar * A^{gamma-1} * sum_j pi * R_port*(j)^{1-gamma}
        omega = 0.0
        for j_s in range(N_state):
            R_s = R_bill * Rx_stock_next[j_s]
            R_b = R_bill * Rx_bond_next[j_s]
            R_p = opt_s * R_s + opt_b * R_b + a_bill * R_bill
            omega += Pi_state[i_s, j_s] * max(R_p, min_return_power) ** (1.0 - gamma)
        omega *= b_bar * A_is ** (gamma - 1.0)

        ratio = (beta * omega) ** (-1.0 / gamma)   # = c*/a* at optimum

        for i_w in range(n_w):
            c_val = max(wealth_grid[i_w] * ratio / (ratio + 1.0), min_consumption)
            for z_i in range(n_z):
                out_c      [z_i, i_s, i_w] = c_val
                out_alpha_s[z_i, i_s, i_w] = opt_s
                out_alpha_b[z_i, i_s, i_w] = opt_b

    return out_c, out_alpha_s, out_alpha_b, terminal_diag_int


# =============================================================================
# PERIOD SOLVER -- RETIREMENT
# =============================================================================

@njit(parallel=True)
def solve_retirement_step(wealth_grid, savings_grid, z_grid, N_state,
                          c_next_full, pension_1d,
                          annuity_factors, Pi_state, mu_r, r_bill_grid,
                          gamma, psi_vec, beta, b_bar,
                          constrained=True, solver_config=None):
    """
    Solve one retirement period using EGM + 2D Newton.
    Parallelised over financial state i_s (prange).

    Parameters
    ----------
    c_next_full  : (n_z, N_state, n_w)  consumption policy at t+1
    pension_1d   : (n_z,)               after-tax pension at t+1
    annuity_factors : (N_state,)         A(y_nom * 4, b_bar) per state
    mu_r         : (N_state, N_state, 2) conditional return means
    r_bill_grid  : (N_state,)            log real bill rate per state

    Returns: policy_c, policy_alpha_s, policy_alpha_b -- each (n_z, N_state, n_w)
             diag_int (N_state, N_DIAG_INT), diag_float (N_state, N_DIAG_FLOAT)
    """

    # --- Solver config ---
    if solver_config is None:
        solver_config = SolverConfig()
    sc = solver_config

    n_z      = len(z_grid)
    n_savings = len(savings_grid)
    n_wealth  = len(wealth_grid)

    policy_c       = np.empty((n_z, N_state, n_wealth))
    policy_alpha_s = np.empty((n_z, N_state, n_wealth))
    policy_alpha_b = np.empty((n_z, N_state, n_wealth))
    diag_int   = np.zeros((N_state, 13), dtype=np.int64)
    diag_float = np.zeros((N_state, 9))

    for i_s in prange(N_state):
        R_bill = exp(r_bill_grid[i_s])
        annuity_factor_is = annuity_factors[i_s]

        # Pre-compute gross excess returns for all next states (avoids re-exp in Newton)
        Rx_stock_next = np.empty(N_state)
        Rx_bond_next  = np.empty(N_state)
        for j_s in range(N_state):
            Rx_stock_next[j_s] = exp(mu_r[i_s, j_s, 0])
            Rx_bond_next[j_s]  = exp(mu_r[i_s, j_s, 1])

        last_a_s = sc.init_alpha_s
        last_a_b = sc.init_alpha_b

        # Init min/max trackers for this i_s
        diag_float[i_s, 3] = 2.0   # DF_MIN_ALPHA_S (init high)
        diag_float[i_s, 5] = 2.0   # DF_MIN_ALPHA_B (init high)

        for z_i in range(n_z):
            psi = psi_vec[z_i]
            c_next_slice    = c_next_full[z_i, :, :]  # (N_state, n_w)
            pension_next    = pension_1d[z_i]

            temp_x = np.empty(n_savings + 1)
            temp_c = np.empty(n_savings + 1)
            temp_s = np.empty(n_savings + 1)
            temp_b = np.empty(n_savings + 1)

            # Anchor at zero savings
            temp_x[0] = sc.egm_anchor;  temp_c[0] = sc.egm_anchor
            temp_s[0] = 0.0;    temp_b[0] = 0.0

            for s_i in range(n_savings):
                s_val = savings_grid[s_i]

                if constrained:
                    opt_s, opt_b, euler, exit_code, foc_resid = solve_portfolio_2d_retirement(
                    s_val, z_i, i_s,
                    wealth_grid, c_next_slice, pension_next,
                    annuity_factor_is, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                    gamma, psi, beta, b_bar,
                    init_s=last_a_s, init_b=last_a_b,
                        tol=sc.tol, max_iter=sc.max_iter,
                        tiny_savings=sc.tiny_savings, corner_tol=sc.corner_tol,
                        edge_max_iter=sc.edge_max_iter, edge_accept_factor=sc.edge_accept_factor,
                        singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                        step_damp=sc.step_damp_constrained, grad_denom_eps=sc.grad_denom_eps,
                        min_wealth_inv=sc.min_wealth_inv, min_consumption=sc.min_consumption,
                        prob_skip=sc.prob_skip_threshold)
                else:
                    opt_s, opt_b, euler, exit_code, foc_resid = solve_portfolio_unconstrained_retirement(
                    s_val, z_i, i_s,
                    wealth_grid, c_next_slice, pension_next,
                    annuity_factor_is, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                    gamma, psi, beta, b_bar,
                    init_s=last_a_s, init_b=last_a_b,
                        tol=sc.tol, max_iter=sc.max_iter_unconstrained,
                        tiny_savings=sc.tiny_savings,
                        singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                        step_damp=sc.step_damp_unconstrained, grad_denom_eps=sc.grad_denom_eps,
                        min_wealth_inv=sc.min_wealth_inv, min_consumption=sc.min_consumption,
                        prob_skip=sc.prob_skip_threshold)

                # -- Diagnostic tracking --
                diag_int[i_s, 10] += 1  # DI_TOTAL_CALLS
                if exit_code == 0:
                    diag_int[i_s, 9] += 1   # DI_TINY_SAVINGS
                elif exit_code == 1:
                    diag_int[i_s, 0] += 1   # DI_CORNER_BILLS
                elif exit_code == 2:
                    diag_int[i_s, 1] += 1   # DI_CORNER_STOCKS
                elif exit_code == 3:
                    diag_int[i_s, 2] += 1   # DI_CORNER_BONDS
                elif exit_code == 4:
                    diag_int[i_s, 3] += 1   # DI_EDGE_SB
                elif exit_code == 5:
                    diag_int[i_s, 4] += 1   # DI_EDGE_BB
                elif exit_code == 6:
                    diag_int[i_s, 5] += 1   # DI_EDGE_STOCKBOND
                elif exit_code == 7:
                    diag_int[i_s, 6] += 1   # DI_INTERIOR
                elif exit_code == 8:
                    diag_int[i_s, 7] += 1   # DI_NEWTON_FAIL

                # FOC residual tracking
                if foc_resid > diag_float[i_s, 1]:  # DF_MAX_FOC_RESID
                    diag_float[i_s, 1] = foc_resid
                diag_float[i_s, 2] += foc_resid * foc_resid  # DF_SUM_FOC_RESID_SQ

                # Portfolio stats
                diag_float[i_s, 7] += opt_s  # DF_SUM_ALPHA_S
                diag_float[i_s, 8] += opt_b  # DF_SUM_ALPHA_B
                if opt_s < diag_float[i_s, 3]:
                    diag_float[i_s, 3] = opt_s  # DF_MIN_ALPHA_S
                if opt_s > diag_float[i_s, 4]:
                    diag_float[i_s, 4] = opt_s  # DF_MAX_ALPHA_S
                if opt_b < diag_float[i_s, 5]:
                    diag_float[i_s, 5] = opt_b  # DF_MIN_ALPHA_B
                if opt_b > diag_float[i_s, 6]:
                    diag_float[i_s, 6] = opt_b  # DF_MAX_ALPHA_B

                # EGM: invert Euler equation for optimal consumption
                if beta * euler <= 0.0:
                    diag_int[i_s, 11] += 1  # DI_NEG_CONSUMPTION
                c_opt = max(beta * euler, sc.euler_inv_floor) ** (-1.0 / gamma)

                temp_x[s_i + 1] = c_opt + s_val
                temp_c[s_i + 1] = c_opt
                temp_s[s_i + 1] = opt_s
                temp_b[s_i + 1] = opt_b

                last_a_s = opt_s
                last_a_b = opt_b

            # EGM monotonicity check
            for s_i in range(n_savings):
                if temp_x[s_i + 1] <= temp_x[s_i]:
                    diag_int[i_s, 12] += 1  # DI_MONO_VIOLATIONS
                    drop = temp_x[s_i] - temp_x[s_i + 1]
                    if drop > diag_float[i_s, 0]:  # DF_WORST_MONO_DROP
                        diag_float[i_s, 0] = drop

            # Interpolate endogenous grid -> exogenous wealth grid
            for w_i in range(n_wealth):
                w = wealth_grid[w_i]
                policy_c      [z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_c)
                policy_alpha_s[z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_s)
                policy_alpha_b[z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_b)

    return policy_c, policy_alpha_s, policy_alpha_b, diag_int, diag_float


# =============================================================================
# PERIOD SOLVER -- WORKING AGE
# =============================================================================

@njit(parallel=True)
def solve_working_age_step(wealth_grid, savings_grid, z_grid, N_state,
                            c_next_full, income_next_table,
                            annuity_factors, Pi_z, Pi_state, mu_r, r_bill_grid,
                            eps_nodes, eps_weights,
                            gamma, psi_vec, beta, b_bar,
                            constrained=True, solver_config=None):
    """
    Solve one working-age period using EGM + 2D Newton.
    Parallelised over financial state i_s (prange).

    Parameters
    ----------
    c_next_full        : (n_z, N_state, n_w)  consumption policy at t+1
    income_next_table  : (n_z, n_eps)          after-tax labor income at t+1
    annuity_factors    : (N_state,)             A(y_nom * 4, b_bar) per state

    Returns: policy_c, policy_alpha_s, policy_alpha_b -- each (n_z, N_state, n_w)
             diag_int (N_state, N_DIAG_INT), diag_float (N_state, N_DIAG_FLOAT)
    """

    # --- Solver config ---
    if solver_config is None:
        solver_config = SolverConfig()
    sc = solver_config

    n_z      = len(z_grid)
    n_savings = len(savings_grid)
    n_wealth  = len(wealth_grid)

    policy_c       = np.empty((n_z, N_state, n_wealth))
    policy_alpha_s = np.empty((n_z, N_state, n_wealth))
    policy_alpha_b = np.empty((n_z, N_state, n_wealth))
    diag_int   = np.zeros((N_state, 13), dtype=np.int64)
    diag_float = np.zeros((N_state, 9))

    for i_s in prange(N_state):
        R_bill = exp(r_bill_grid[i_s])
        annuity_factor_is = annuity_factors[i_s]

        Rx_stock_next = np.empty(N_state)
        Rx_bond_next  = np.empty(N_state)
        for j_s in range(N_state):
            Rx_stock_next[j_s] = exp(mu_r[i_s, j_s, 0])
            Rx_bond_next[j_s]  = exp(mu_r[i_s, j_s, 1])

        last_a_s = sc.init_alpha_s
        last_a_b = sc.init_alpha_b

        # Init min/max trackers for this i_s
        diag_float[i_s, 3] = 2.0   # DF_MIN_ALPHA_S (init high)
        diag_float[i_s, 5] = 2.0   # DF_MIN_ALPHA_B (init high)

        for z_i in range(n_z):
            psi = psi_vec[z_i]
            temp_x = np.empty(n_savings + 1)
            temp_c = np.empty(n_savings + 1)
            temp_s = np.empty(n_savings + 1)
            temp_b = np.empty(n_savings + 1)

            temp_x[0] = sc.egm_anchor;  temp_c[0] = sc.egm_anchor
            temp_s[0] = 0.0;    temp_b[0] = 0.0

            for s_i in range(n_savings):
                s_val = savings_grid[s_i]

                if constrained:
                    opt_s, opt_b, euler, exit_code, foc_resid = solve_portfolio_2d_working(
                    s_val, z_i, i_s,
                    wealth_grid, c_next_full, income_next_table,
                    annuity_factor_is, Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                    eps_nodes, eps_weights, gamma, psi, beta, b_bar,
                    init_s=last_a_s, init_b=last_a_b,
                        tol=sc.tol, max_iter=sc.max_iter,
                        tiny_savings=sc.tiny_savings, corner_tol=sc.corner_tol,
                        edge_max_iter=sc.edge_max_iter, edge_accept_factor=sc.edge_accept_factor,
                        singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                        step_damp=sc.step_damp_constrained, grad_denom_eps=sc.grad_denom_eps,
                        min_wealth_inv=sc.min_wealth_inv, min_consumption=sc.min_consumption,
                        prob_skip=sc.prob_skip_threshold)
                else:
                    opt_s, opt_b, euler, exit_code, foc_resid = solve_portfolio_unconstrained_working(
                    s_val, z_i, i_s,
                    wealth_grid, c_next_full, income_next_table,
                    annuity_factor_is, Pi_z, Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                    eps_nodes, eps_weights, gamma, psi, beta, b_bar,
                    init_s=last_a_s, init_b=last_a_b,
                        tol=sc.tol, max_iter=sc.max_iter_unconstrained,
                        tiny_savings=sc.tiny_savings,
                        singular_det=sc.singular_det, grad_step_size=sc.grad_step_size,
                        step_damp=sc.step_damp_unconstrained, grad_denom_eps=sc.grad_denom_eps,
                        min_wealth_inv=sc.min_wealth_inv, min_consumption=sc.min_consumption,
                        prob_skip=sc.prob_skip_threshold)

                # -- Diagnostic tracking --
                diag_int[i_s, 10] += 1  # DI_TOTAL_CALLS
                if exit_code == 0:
                    diag_int[i_s, 9] += 1
                elif exit_code == 1:
                    diag_int[i_s, 0] += 1
                elif exit_code == 2:
                    diag_int[i_s, 1] += 1
                elif exit_code == 3:
                    diag_int[i_s, 2] += 1
                elif exit_code == 4:
                    diag_int[i_s, 3] += 1
                elif exit_code == 5:
                    diag_int[i_s, 4] += 1
                elif exit_code == 6:
                    diag_int[i_s, 5] += 1
                elif exit_code == 7:
                    diag_int[i_s, 6] += 1
                elif exit_code == 8:
                    diag_int[i_s, 7] += 1

                if foc_resid > diag_float[i_s, 1]:
                    diag_float[i_s, 1] = foc_resid
                diag_float[i_s, 2] += foc_resid * foc_resid

                diag_float[i_s, 7] += opt_s
                diag_float[i_s, 8] += opt_b
                if opt_s < diag_float[i_s, 3]:
                    diag_float[i_s, 3] = opt_s
                if opt_s > diag_float[i_s, 4]:
                    diag_float[i_s, 4] = opt_s
                if opt_b < diag_float[i_s, 5]:
                    diag_float[i_s, 5] = opt_b
                if opt_b > diag_float[i_s, 6]:
                    diag_float[i_s, 6] = opt_b

                if beta * euler <= 0.0:
                    diag_int[i_s, 11] += 1

                c_opt = max(beta * euler, sc.euler_inv_floor) ** (-1.0 / gamma)

                temp_x[s_i + 1] = c_opt + s_val
                temp_c[s_i + 1] = c_opt
                temp_s[s_i + 1] = opt_s
                temp_b[s_i + 1] = opt_b

                last_a_s = opt_s
                last_a_b = opt_b

            # EGM monotonicity check
            for s_i in range(n_savings):
                if temp_x[s_i + 1] <= temp_x[s_i]:
                    diag_int[i_s, 12] += 1
                    drop = temp_x[s_i] - temp_x[s_i + 1]
                    if drop > diag_float[i_s, 0]:
                        diag_float[i_s, 0] = drop

            for w_i in range(n_wealth):
                w = wealth_grid[w_i]
                policy_c      [z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_c)
                policy_alpha_s[z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_s)
                policy_alpha_b[z_i, i_s, w_i] = fast_interp_1d(w, temp_x, temp_b)

    return policy_c, policy_alpha_s, policy_alpha_b, diag_int, diag_float


# =============================================================================
# MASTER SOLVER
# =============================================================================

def _reduce_diag(diag_int, diag_float):
    """Reduce per-i_s diagnostic arrays to totals."""
    ti = diag_int.sum(axis=0)
    tf_sum = diag_float.sum(axis=0)
    tf_max = diag_float.max(axis=0)
    tf_min = diag_float.min(axis=0)
    return ti, tf_sum, tf_max, tf_min


def _format_pct(count, total):
    if total == 0:
        return "  0%"
    return f"{100.0 * count / total:3.0f}%"


def run_lifecycle_solver(model, pc, n_s_points=None, solver_config=None, verbose=1):
    """
    Lifecycle backward induction solver.

    Parameters
    ----------
    model : LifecyclePortfolioModel
    pc    : Precompute
    n_s_points : int, optional  -- override savings grid size
    verbose : int  -- 0=silent, 1=per-age table + post-solve report (default)

    Returns
    -------
    C_mat, S_mat, B_mat : np.ndarray, shape (n_age, n_z, N_state, n_w)
        Optimal consumption, stock share, and bond share.
    diagnostics : dict
        Diagnostic summary from the solve.
    """
    if verbose >= 1:
        print(f"\n{'='*70}")
        print(f"LIFECYCLE PORTFOLIO SOLVER  (EGM + 2D Newton)")
        mode_str = "CONSTRAINED" if model.constrained else "UNCONSTRAINED"
        print(f"  Mode: {mode_str}")
        print(f"  Solver: {solver_config}")
        print(f"  Discretization: {pc.disc_config}")
        print(f"{'='*70}")

    # ---- Grids ----
    w_grid = pc.wealth_grid
    s_grid = pc.s_grid if n_s_points is None else pc.regenerate_savings_grid(n_points=n_s_points)
    z_grid = pc.z_grid
    ages   = pc.ages

    n_w     = len(w_grid)
    n_z     = pc.n_z
    N_state = pc.N_state
    n_age   = pc.n_age

    # ---- Transitions and returns ----
    Pi_state        = pc.Pi_state
    Pi_z            = pc.Pi_z
    mu_r            = pc.mu_r
    r_bill_grid     = pc.r_bill_grid
    annuity_factors = pc.annuity_factors

    # ---- Income tables ----
    pension_table        = pc.pension_after_tax      # (n_age, n_z)
    working_income_table = pc.working_income          # (n_age, n_z, n_eps)
    eps_nodes   = pc.eps_nodes
    eps_weights = pc.eps_weights

    # ---- Model parameters ----
    gamma          = model.gamma
    beta           = model.beta
    b_bar          = model.b_bar
    survival_probs = pc.survival_probs_2d   # (n_age, n_z)
    retire_age     = model.retire_age
    start_age      = model.start_age
    terminal_age   = model.terminal_age
    constrained    = model.constrained

    if solver_config is None:
        solver_config = SolverConfig()

    if verbose >= 1:
        print(f"  Ages {start_age}\u2013{terminal_age}  ({n_age} periods)")
        print(f"  Grids: n_w={n_w}, n_s={len(s_grid)}, n_z={n_z}, N_state={N_state}")
        print(f"  gamma={gamma}, beta={beta}, b_bar={b_bar}")
        print(f"  States per period: {n_z} \u00d7 {N_state} = {n_z * N_state:,}")

    # ---- Median indices for per-age summary ----
    i_z_med = n_z // 2
    i_s_med = N_state // 2
    i_w_med = n_w // 2

    # ---- Policy arrays ----
    shape = (n_age, n_z, N_state, n_w)
    C_mat = np.zeros(shape)
    S_mat = np.zeros(shape)
    B_mat = np.zeros(shape)

    # ---- Per-age diagnostic accumulators ----
    age_diag_int     = np.zeros((n_age, N_DIAG_INT), dtype=np.int64)
    age_diag_fsum    = np.zeros((n_age, N_DIAG_FLOAT))
    age_diag_fmax    = np.zeros((n_age, N_DIAG_FLOAT))
    age_diag_fmin    = np.full((n_age, N_DIAG_FLOAT), np.inf)

    # ---- Terminal condition ----
    if verbose >= 1:
        print(f"\n  Terminal condition (age {terminal_age}) ... ", end="", flush=True)
    c_T, a_s_T, a_b_T, term_diag = solve_terminal_age(
        w_grid, annuity_factors, r_bill_grid, Pi_state, mu_r,
        gamma, beta, b_bar, N_state, n_z, constrained=constrained, solver_config=solver_config)
    C_mat[-1] = c_T
    S_mat[-1] = a_s_T
    B_mat[-1] = a_b_T
    if verbose >= 1:
        n_term_interior = int(np.sum(term_diag == EC_INTERIOR))
        n_term_fail = int(np.sum(term_diag == EC_NEWTON_FAIL))
        print(f"done  [c range: {c_T.min():.3f}\u2013{c_T.max():.3f}]  "
              f"[portfolio: {n_term_interior} interior, {N_state - n_term_interior - n_term_fail} corner/edge"
              f"{f', {n_term_fail} FAIL' if n_term_fail > 0 else ''}]")

    # ---- Backward induction ----
    if verbose >= 1:
        print(f"\n{'='*120}")
        hdr = (f" {'Age':>3}  {'Phase':<6} {'Time':>5}  {'Newt%':>5} {'Fail':>6}"
               f"  {'alpha_s':>7}  {'alpha_b':>7}  {'a_bill':>7}  {'c/W':>5}"
               f"  {'%int':>4}  {'%edge':>5}  {'%corn':>5}  {'mono':>4}")
        print(hdr)
        print(f"{'='*120}")

    t_start = time.time()

    for t in reversed(range(n_age - 1)):
        age    = ages[t]
        psi    = survival_probs[t, :]      # (n_z,) -- z-dependent survival
        c_next = C_mat[t + 1]

        if age >= retire_age:
            c, a_s, a_b, _di, _df = solve_retirement_step(
                w_grid, s_grid, z_grid, N_state,
                c_next, pension_table[t + 1, :],
                annuity_factors, Pi_state, mu_r, r_bill_grid,
                gamma, psi, beta, b_bar, constrained=constrained, solver_config=solver_config)
            label = "RETIRE"
        else:
            c, a_s, a_b, _di, _df = solve_working_age_step(
                w_grid, s_grid, z_grid, N_state,
                c_next, working_income_table[t + 1, :, :],
                annuity_factors, Pi_z, Pi_state, mu_r, r_bill_grid,
                eps_nodes, eps_weights,
                gamma, psi, beta, b_bar, constrained=constrained, solver_config=solver_config)
            label = "WORK  "

        C_mat[t] = c
        S_mat[t] = a_s
        B_mat[t] = a_b

        # Reduce diagnostics for this age
        ti, tf_sum, tf_max, tf_min = _reduce_diag(_di, _df)
        age_diag_int[t]  = ti
        age_diag_fsum[t] = tf_sum
        age_diag_fmax[t] = tf_max
        age_diag_fmin[t] = tf_min

        # Per-age one-line summary
        if verbose >= 1:
            elapsed = time.time() - t_start
            total_calls = int(ti[DI_TOTAL_CALLS])
            n_fail = int(ti[DI_NEWTON_FAIL])
            newton_pct = 100.0 * (total_calls - n_fail) / max(total_calls, 1)

            n_interior = int(ti[DI_INTERIOR])
            n_edge = int(ti[DI_EDGE_SB] + ti[DI_EDGE_BB] + ti[DI_EDGE_STOCKBOND])
            n_corner = int(ti[DI_CORNER_BILLS] + ti[DI_CORNER_STOCKS] + ti[DI_CORNER_BONDS]
                           + ti[DI_TINY_SAVINGS])
            mono_v = int(ti[DI_MONO_VIOLATIONS])

            # Median-state policy values
            med_as = float(a_s[i_z_med, i_s_med, i_w_med])
            med_ab = float(a_b[i_z_med, i_s_med, i_w_med])
            med_bill = 1.0 - med_as - med_ab
            med_c = float(c[i_z_med, i_s_med, i_w_med])
            med_w = float(w_grid[i_w_med])
            c_over_w = med_c / med_w if med_w > 0 else 0.0

            mono_str = f"{mono_v:4d}" if mono_v == 0 else f"\033[91m{mono_v:4d}\033[0m"

            print(f" {age:3d}  {label:<6} {elapsed:5.1f}s  {newton_pct:5.1f}% {n_fail:>6}"
                  f"  {med_as:7.3f}  {med_ab:7.3f}  {med_bill:7.3f}  {c_over_w:5.3f}"
                  f"  {_format_pct(n_interior, total_calls)}"
                  f"  {_format_pct(n_edge, total_calls)}"
                  f"  {_format_pct(n_corner, total_calls)}"
                  f"  {mono_str}", flush=True)

    total = time.time() - t_start

    # ========================================================================
    # POST-SOLVE DIAGNOSTICS
    # ========================================================================

    # Aggregate across all ages (exclude terminal t=-1 which has no diag arrays)
    all_int = age_diag_int[:-1].sum(axis=0)  # sum across ages
    all_fsum = age_diag_fsum[:-1].sum(axis=0)
    all_fmax = age_diag_fmax[:-1].max(axis=0)
    all_fmin = age_diag_fmin[:-1].min(axis=0)

    total_calls = int(all_int[DI_TOTAL_CALLS])
    total_fail = int(all_int[DI_NEWTON_FAIL])
    total_mono = int(all_int[DI_MONO_VIOLATIONS])
    worst_mono = float(all_fmax[DF_WORST_MONO_DROP])
    worst_foc = float(all_fmax[DF_MAX_FOC_RESID])
    rms_foc = (all_fsum[DF_SUM_FOC_RESID_SQ] / max(total_calls, 1)) ** 0.5

    # Build diagnostics dict
    diagnostics = {
        'age_diag_int': age_diag_int,
        'age_diag_fsum': age_diag_fsum,
        'age_diag_fmax': age_diag_fmax,
        'age_diag_fmin': age_diag_fmin,
        'total_mono_violations': total_mono,
        'worst_mono_drop': worst_mono,
        'total_newton_failures': total_fail,
        'worst_foc_resid': worst_foc,
        'total_calls': total_calls,
        'constrained': constrained,
        'solver_config': solver_config,
        'disc_config': pc.disc_config,
    }

    if verbose >= 1:
        print(f"\n{'='*120}")
        print(f"  DONE in {total / 60:.2f} min  (avg {total / max(n_age - 1, 1):.2f}s per age)")
        print(f"{'='*120}")

        # --- Section 1: Newton Convergence ---
        print(f"\n{'='*70}")
        print(f"  POST-SOLVE DIAGNOSTICS")
        print(f"{'='*70}")

        print(f"\n  1. NEWTON CONVERGENCE")
        print(f"     Total calls:  {total_calls:>12,}")
        print(f"     Converged:    {total_calls - total_fail:>12,}  ({100.0 * (total_calls - total_fail) / max(total_calls, 1):.3f}%)")
        print(f"     Failed:       {total_fail:>12,}  ({100.0 * total_fail / max(total_calls, 1):.3f}%)")
        print(f"     Worst FOC:    {worst_foc:>12.2e}")
        print(f"     RMS FOC:      {rms_foc:>12.2e}")
        if total_fail > 0:
            print(f"\n     Ages with failures:")
            for t in range(n_age - 1):
                nf = int(age_diag_int[t, DI_NEWTON_FAIL])
                if nf > 0:
                    age = ages[t]
                    lbl = "RETIRE" if age >= retire_age else "WORK"
                    mfoc = float(age_diag_fmax[t, DF_MAX_FOC_RESID])
                    print(f"       Age {age:3d} {lbl:>6}: {nf:4d} failures  (max resid {mfoc:.2e})")

        # --- Section 2: Portfolio Regime Breakdown ---
        print(f"\n  2. PORTFOLIO REGIME BREAKDOWN")
        retire_mask = np.array([ages[t] >= retire_age for t in range(n_age - 1)])
        work_mask   = ~retire_mask

        def _regime_row(mask, label):
            sel = age_diag_int[:-1][mask].sum(axis=0) if mask.any() else np.zeros(N_DIAG_INT, dtype=np.int64)
            tot = int(sel[DI_TOTAL_CALLS])
            if tot == 0:
                return
            bills  = int(sel[DI_CORNER_BILLS] + sel[DI_TINY_SAVINGS])
            stocks = int(sel[DI_CORNER_STOCKS])
            bonds  = int(sel[DI_CORNER_BONDS])
            sb     = int(sel[DI_EDGE_SB])
            bb     = int(sel[DI_EDGE_BB])
            sB     = int(sel[DI_EDGE_STOCKBOND])
            intr   = int(sel[DI_INTERIOR])
            fail   = int(sel[DI_NEWTON_FAIL])
            print(f"     {label:<12}"
                  f"  {100*bills/tot:5.1f}%"
                  f"  {100*stocks/tot:5.1f}%"
                  f"  {100*bonds/tot:5.1f}%"
                  f"  {100*sb/tot:5.1f}%"
                  f"  {100*bb/tot:5.1f}%"
                  f"  {100*sB/tot:5.1f}%"
                  f"  {100*intr/tot:5.1f}%"
                  f"  {100*fail/tot:5.1f}%")

        print(f"     {'':12}  {'Bills':>6}  {'Stocks':>6}  {'Bonds':>6}"
              f"  {'S+Bill':>6}  {'B+Bill':>6}  {'S+Bond':>6}  {'Inter.':>6}  {'Fail':>6}")
        _regime_row(retire_mask, "Retirement:")
        _regime_row(work_mask,   "Working:")
        _regime_row(np.ones(n_age - 1, dtype=bool), "Overall:")

        # --- Section 3: Portfolio Share Ranges ---
        print(f"\n  3. PORTFOLIO SHARE RANGES")
        mean_as = all_fsum[DF_SUM_ALPHA_S] / max(total_calls, 1)
        mean_ab = all_fsum[DF_SUM_ALPHA_B] / max(total_calls, 1)
        print(f"     Stock:  [{all_fmin[DF_MIN_ALPHA_S]:.3f}, {all_fmax[DF_MAX_ALPHA_S]:.3f}]  mean={mean_as:.3f}")
        print(f"     Bond:   [{all_fmin[DF_MIN_ALPHA_B]:.3f}, {all_fmax[DF_MAX_ALPHA_B]:.3f}]  mean={mean_ab:.3f}")
        print(f"     Bill:   mean={1.0 - mean_as - mean_ab:.3f}  (inferred: 1-s-b)")

        # --- Section 4: EGM Monotonicity ---
        print(f"\n  4. EGM MONOTONICITY")
        if total_mono > 0:
            n_affected_ages = int(np.sum(age_diag_int[:-1, DI_MONO_VIOLATIONS] > 0))
            print(f"     WARNING: {total_mono} total violations across {n_affected_ages}/{n_age-1} ages")
            print(f"     Worst drop: {worst_mono:.2e}")
            for t in range(n_age - 1):
                mv = int(age_diag_int[t, DI_MONO_VIOLATIONS])
                if mv > 0:
                    age = ages[t]
                    lbl = "RETIRE" if age >= retire_age else "WORK"
                    wd = float(age_diag_fmax[t, DF_WORST_MONO_DROP])
                    print(f"       Age {age:3d} {lbl:>6}: {mv:4d} violations, worst drop {wd:.2e}")
        else:
            print(f"     PASS")

        # --- Section 5: Policy Function Sanity ---
        print(f"\n  5. POLICY FUNCTION SANITY")
        nan_c = int(np.isnan(C_mat).sum())
        nan_s = int(np.isnan(S_mat).sum())
        nan_b = int(np.isnan(B_mat).sum())
        inf_c = int(np.isinf(C_mat).sum())
        inf_s = int(np.isinf(S_mat).sum())
        inf_b = int(np.isinf(B_mat).sum())
        neg_c = int((C_mat < 0).sum())
        neg_euler = int(all_int[DI_NEG_CONSUMPTION])
        alpha_s_neg = int((S_mat < -1e-6).sum())
        alpha_b_neg = int((B_mat < -1e-6).sum())
        alpha_sum_viol = int(((S_mat + B_mat) > 1.0 + 1e-6).sum())
        total_el = C_mat.size + S_mat.size + B_mat.size

        if constrained:
            all_ok = (nan_c + nan_s + nan_b + inf_c + inf_s + inf_b
                      + neg_c + alpha_s_neg + alpha_b_neg + alpha_sum_viol + neg_euler == 0)
        else:
            # Unconstrained: negative alphas and sum > 1 are expected
            all_ok = (nan_c + nan_s + nan_b + inf_c + inf_s + inf_b
                      + neg_c + neg_euler == 0)
        if all_ok:
            print(f"     PASS  ({total_el:,} elements checked)")
        else:
            print(f"     NaN count:       C={nan_c}  S={nan_s}  B={nan_b}")
            print(f"     Inf count:       C={inf_c}  S={inf_s}  B={inf_b}")
            print(f"     C < 0:           {neg_c}")
            print(f"     Neg euler:       {neg_euler}")
            print(f"     alpha_s < -1e-6: {alpha_s_neg}")
            print(f"     alpha_b < -1e-6: {alpha_b_neg}")
            print(f"     alpha_s+b > 1:   {alpha_sum_viol}")

        # --- Section 6: Consumption Profile ---
        print(f"\n  6. CONSUMPTION PROFILE (c/W at median state, median wealth)")
        cw_profile = np.zeros(n_age)
        for t in range(n_age):
            c_val = float(C_mat[t, i_z_med, i_s_med, i_w_med])
            w_val = float(w_grid[i_w_med])
            cw_profile[t] = c_val / w_val if w_val > 0 else 0.0

        cw_min_t = int(np.argmin(cw_profile))
        cw_max_t = int(np.argmax(cw_profile))
        print(f"     Min c/W: {cw_profile[cw_min_t]:.4f} (age {ages[cw_min_t]})")
        print(f"     Max c/W: {cw_profile[cw_max_t]:.4f} (age {ages[cw_max_t]})")

        # Check monotonicity of c/W profile
        cw_dips = []
        for t in range(1, n_age):
            if cw_profile[t] < cw_profile[t-1] - 1e-6:
                cw_dips.append((ages[t], cw_profile[t-1] - cw_profile[t]))
        if cw_dips:
            print(f"     WARNING: c/W non-monotone at {len(cw_dips)} age(s):")
            for age_d, drop_d in cw_dips[:5]:
                print(f"       Age {age_d}: drop = {drop_d:.4f}")
        else:
            print(f"     c/W monotonically increasing: PASS")

        print(f"{'='*70}\n")

    return C_mat, S_mat, B_mat, diagnostics


In [24]:
# Run the lifecycle solver
# First call triggers Numba JIT compilation (~30-60s); subsequent calls are fast.
C_mat, S_mat, B_mat, diagnostics = run_lifecycle_solver(model, pc, verbose=1)

print(f"\nPolicy array shapes: {C_mat.shape}")
print(f"  C_mat[t, i_z, i_s, i_w] = consumption")
print(f"  S_mat[t, i_z, i_s, i_w] = stock share (alpha_stock)")
print(f"  B_mat[t, i_z, i_s, i_w] = bond share  (alpha_bond)")
print(f"\nDiagnostics keys: {list(diagnostics.keys())}")


LIFECYCLE PORTFOLIO SOLVER  (EGM + 2D Newton)
  Ages 25–80  (56 periods)
  Grids: n_w=150, n_s=150, n_z=11, N_state=125
  gamma=3.0, beta=0.96, b_bar=10
  States per period: 11 × 125 = 1,375

  Terminal condition (age 80) ... done  [c range: 0.000–24.438]  [portfolio: 5 interior, 120 corner/edge]

 Age  Phase   Time  Newt%   Fail  alpha_s  alpha_b   a_bill    c/W  %int  %edge  %corn  mono
  79  RETIRE   4.1s   93.6%  13272    0.457    0.329    0.214  0.138    1%   14%   79%     0
  78  RETIRE   5.1s   93.6%  13244    0.452    0.322    0.227  0.142    1%   14%   79%     0
  77  RETIRE   6.0s   93.6%  13138    0.452    0.322    0.227  0.147    1%   14%   79%     0
  76  RETIRE   6.9s   93.7%  12978    0.452    0.322    0.225  0.152    1%   14%   79%     0
  75  RETIRE   7.9s   93.7%  12943    0.453    0.323    0.223  0.159    1%   14%   79%     0
  74  RETIRE   8.8s   93.8%  12819    0.454    0.325    0.221  0.168    1%   14%   79%     0
  73  RETIRE   9.7s   93.9%  12668    0.456    0.

In [27]:
# =============================================================================
# NEWTON FAILURE DIAGNOSTICS
# =============================================================================
# Investigate WHY ~20% of Newton calls fail.
# Strategy: pick one retirement age, sweep all (i_s, z_i, s_i) for that age,
# evaluate FOCs at corners, and classify why each call fails.

from numba import njit
from math import exp

@njit
def diagnose_newton_failures_retirement(
    wealth_grid, savings_grid, z_grid, N_state,
    c_next_full, pension_1d,
    annuity_factors, Pi_state, mu_r, r_bill_grid,
    gamma, psi_vec, beta, b_bar):
    """
    For each (i_s, z_i, s_i), evaluate corner FOCs and classify the failure mode.
    Returns per-i_s diagnostics (not parallelized, for clarity).
    
    Returns:
        corner_focs : (N_state, 6) -- [fs0, fb0, fs1, fb1, fs2, fb2] at median z, median s
        failure_reasons : (N_state, 6) -- counts per i_s:
            [0] = no_bracket_any_edge (corners rejected, no edge bracket exists)
            [1] = edge_newton_didnt_converge (bracket exists but edge Newton failed acceptance)
            [2] = interior_newton_fail (fell through to interior, didn't converge)
            [3] = total_fail_calls (total Newton failures for this i_s)
            [4] = total_calls (total calls for this i_s)
            [5] = edge_tried_but_rejected (edge Newton ran but acceptance check failed)
    """
    n_z = len(z_grid)
    n_savings = len(savings_grid)
    
    corner_focs = np.empty((N_state, 6))
    failure_reasons = np.zeros((N_state, 6), dtype=np.int64)
    
    z_med = n_z // 2
    s_med = n_savings // 2
    
    for i_s in range(N_state):
        psi_med = psi_vec[z_med]
        R_bill = exp(r_bill_grid[i_s])
        annuity_factor_is = annuity_factors[i_s]
        
        Rx_stock_next = np.empty(N_state)
        Rx_bond_next = np.empty(N_state)
        for j_s in range(N_state):
            Rx_stock_next[j_s] = exp(mu_r[i_s, j_s, 0])
            Rx_bond_next[j_s] = exp(mu_r[i_s, j_s, 1])
        
        # Evaluate corner FOCs at median savings for reporting
        s_val_med = savings_grid[s_med]
        c_next_slice = c_next_full[z_med, :, :]
        pension_next = pension_1d[z_med]
        
        fs0, fb0, _, _, _, _ = compute_foc_jac_retirement(
            0.0, 0.0, s_val_med, z_med, i_s,
            wealth_grid, c_next_slice, pension_next, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi_med, beta, b_bar)
        fs1, fb1, _, _, _, _ = compute_foc_jac_retirement(
            1.0, 0.0, s_val_med, z_med, i_s,
            wealth_grid, c_next_slice, pension_next, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi_med, beta, b_bar)
        fs2, fb2, _, _, _, _ = compute_foc_jac_retirement(
            0.0, 1.0, s_val_med, z_med, i_s,
            wealth_grid, c_next_slice, pension_next, annuity_factor_is,
            Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi_med, beta, b_bar)
        
        corner_focs[i_s, 0] = fs0
        corner_focs[i_s, 1] = fb0
        corner_focs[i_s, 2] = fs1
        corner_focs[i_s, 3] = fb1
        corner_focs[i_s, 4] = fs2
        corner_focs[i_s, 5] = fb2
        
        # Now sweep all (z_i, s_i) and count failure modes
        last_a_s = 0.1
        last_a_b = 0.4
        for z_i in range(n_z):
            psi = psi_vec[z_i]
            c_slice = c_next_full[z_i, :, :]
            pens = pension_1d[z_i]
            for s_i in range(n_savings):
                s_val = savings_grid[s_i]
                failure_reasons[i_s, 4] += 1  # total_calls
                
                _, _, _, exit_code, foc_resid = solve_portfolio_2d_retirement(
                    s_val, z_i, i_s,
                    wealth_grid, c_slice, pens, annuity_factor_is,
                    Pi_state, Rx_stock_next, Rx_bond_next, R_bill,
                    gamma, psi, beta, b_bar,
                    init_s=last_a_s, init_b=last_a_b)
                
                if exit_code == EC_NEWTON_FAIL:
                    failure_reasons[i_s, 3] += 1  # total_fail_calls
                    failure_reasons[i_s, 2] += 1  # interior_newton_fail
                    
                    # WHY did it fail? Re-evaluate corners for this specific (z_i, s_i)
                    _fs0, _fb0, _, _, _, _ = compute_foc_jac_retirement(
                        0.0, 0.0, s_val, z_i, i_s,
                        wealth_grid, c_slice, pens, annuity_factor_is,
                        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar)
                    _fs1, _fb1, _, _, _, _ = compute_foc_jac_retirement(
                        1.0, 0.0, s_val, z_i, i_s,
                        wealth_grid, c_slice, pens, annuity_factor_is,
                        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar)
                    _fs2, _fb2, _, _, _, _ = compute_foc_jac_retirement(
                        0.0, 1.0, s_val, z_i, i_s,
                        wealth_grid, c_slice, pens, annuity_factor_is,
                        Pi_state, Rx_stock_next, Rx_bond_next, R_bill, gamma, psi, beta, b_bar)
                    
                    # Check bracket conditions
                    has_sb = (_fs0 > 0.0 and _fs1 < 0.0)
                    has_bb = (_fb0 > 0.0 and _fb2 < 0.0)
                    _g1 = _fs1 - _fb1
                    _g2 = _fs2 - _fb2
                    has_stockbond = (_g1 * _g2 < 0.0)
                    
                    if not has_sb and not has_bb and not has_stockbond:
                        failure_reasons[i_s, 0] += 1  # no_bracket_any_edge
                    else:
                        failure_reasons[i_s, 5] += 1  # edge_tried_but_rejected
    
    return corner_focs, failure_reasons


# --- Run the diagnostic on age 79 (first retirement age solved) ---
t_diag = pc.n_age - 2  # age 79
age_diag = pc.ages[t_diag]
psi_diag = pc.survival_probs_2d[t_diag, :]   # (n_z,)
c_next_diag = C_mat[t_diag + 1]  # continuation value from age 80

print(f"Diagnosing Newton failures at age {age_diag} (t={t_diag}, psi range=[{psi_diag.min():.4f}, {psi_diag.max():.4f}])")
print(f"Continuation value from age {pc.ages[t_diag+1]}")
print()

corner_focs, fail_reasons = diagnose_newton_failures_retirement(
    pc.wealth_grid, pc.s_grid, pc.z_grid, pc.N_state,
    c_next_diag, pc.pension_after_tax[t_diag + 1, :],
    pc.annuity_factors, pc.Pi_state, pc.mu_r, pc.r_bill_grid,
    model.gamma, psi_diag, model.beta, model.b_bar)

# --- Summary ---
total_fails = fail_reasons[:, 3].sum()
total_calls = fail_reasons[:, 4].sum()
n_failing_states = np.sum(fail_reasons[:, 3] > 0)

print(f"Total calls: {total_calls:,}   Failures: {total_fails:,}  ({100*total_fails/total_calls:.1f}%)")
print(f"Failing financial states: {n_failing_states} / {pc.N_state}")
print()

# Failure mode breakdown
no_bracket = fail_reasons[:, 0].sum()
edge_rejected = fail_reasons[:, 5].sum()
print(f"Failure mode breakdown (across all failing calls):")
print(f"  No edge bracket exists:     {no_bracket:>8,}  ({100*no_bracket/max(total_fails,1):.1f}%)")
print(f"  Edge tried but rejected:    {edge_rejected:>8,}  ({100*edge_rejected/max(total_fails,1):.1f}%)")
print(f"  (Both types fall to interior Newton which then fails)")
print()

# Per-state detail for failing states
print(f"{'i_s':>4}  {'fails':>6}  {'calls':>6}  {'fail%':>5}  {'no_brk':>6}  {'edge_rej':>8}"
      f"  {'fs0':>10}  {'fb0':>10}  {'fs1':>10}  {'fb1':>10}  {'fs2':>10}  {'fb2':>10}"
      f"  {'state values':>30}")
print("-" * 150)

# Get state variable names and grid for context
state_names = list(model.state_names)
state_grid = pc.state_grid  # (N_state, n_state)

for i_s in range(pc.N_state):
    if fail_reasons[i_s, 3] > 0:
        nf = fail_reasons[i_s, 3]
        nc = fail_reasons[i_s, 4]
        nb = fail_reasons[i_s, 0]
        er = fail_reasons[i_s, 5]
        f0, f1, f2, f3, f4, f5 = corner_focs[i_s]
        sv = state_grid[i_s]
        sv_str = "  ".join(f"{state_names[k]}={sv[k]:+.4f}" for k in range(len(state_names)))
        print(f"{i_s:4d}  {nf:6d}  {nc:6d}  {100*nf/nc:4.1f}%  {nb:6d}  {er:8d}"
              f"  {f0:10.2e}  {f1:10.2e}  {f2:10.2e}  {f3:10.2e}  {f4:10.2e}  {f5:10.2e}"
              f"  {sv_str}")

# --- Bracket analysis ---
print(f"\n{'='*70}")
print("BRACKET ANALYSIS for failing states (at median z, median s):")
print(f"{'='*70}")
print(f"  Bracket S+Bill (fs0>0 & fs1<0):  stock FOC changes sign from bills to all-stocks")
print(f"  Bracket B+Bill (fb0>0 & fb2<0):  bond FOC changes sign from bills to all-bonds")
print(f"  Bracket S+Bond (g1*g2<0):        excess stock-vs-bond FOC changes sign")
print()
for i_s in range(pc.N_state):
    if fail_reasons[i_s, 3] > 0:
        f0, f1, f2, f3, f4, f5 = corner_focs[i_s]
        sb = "YES" if (f0 > 0 and f2 < 0) else " no"
        bb = "YES" if (f1 > 0 and f5 < 0) else " no"
        g1 = f2 - f3
        g2 = f4 - f5
        stockbond = "YES" if (g1 * g2 < 0) else " no"
        sv = state_grid[i_s]
        print(f"  i_s={i_s:3d}  S+Bill:{sb}  B+Bill:{bb}  S+Bond:{stockbond}"
              f"  | fs0={f0:+.2e} fb0={f1:+.2e} fs1={f2:+.2e} fb1={f3:+.2e} fs2={f4:+.2e} fb2={f5:+.2e}")


Diagnosing Newton failures at age 79 (t=54, psi=0.6364)
Continuation value from age 80

Total calls: 206,250   Failures: 13,271  (6.4%)
Failing financial states: 33 / 125

Failure mode breakdown (across all failing calls):
  No edge bracket exists:            0  (0.0%)
  Edge tried but rejected:      13,271  (100.0%)
  (Both types fall to interior Newton which then fails)

 i_s   fails   calls  fail%  no_brk  edge_rej         fs0         fb0         fs1         fb1         fs2         fb2                    state values
------------------------------------------------------------------------------------------------------------------------------------------------------
   1     476    1650  28.8%       0       476    4.71e+09   -5.63e+09   -1.28e+09   -5.63e+09    5.69e+09   -1.31e+10  rtb=-0.0142  y_nom=-0.0019  dp=-4.3856
   6     465    1650  28.2%       0       465    7.23e+08   -1.76e+09   -4.15e+09   -2.01e+09    6.68e+08   -6.11e+09  rtb=-0.0142  y_nom=+0.0036  dp=-4.3856
  10   

In [26]:
# =============================================================================
# STEP 1 DIAGNOSTIC: Terminal failures, return magnitudes, continuation values
# =============================================================================
# Requires: model, pc, C_mat, S_mat, B_mat (from cell 11 solve)
#           corner_focs, fail_reasons (from cell 12 diagnostic)

import numpy as np

# Re-run terminal solve to get term_diag (fast, no backward induction)
_, _, _, term_diag = solve_terminal_age(
    pc.wealth_grid, pc.annuity_factors, pc.r_bill_grid, pc.Pi_state, pc.mu_r,
    model.gamma, model.beta, model.b_bar, pc.N_state, pc.n_z)

# -- 1A. Which terminal states (age 80) failed? ------------------------------

failed_terminal = np.where(term_diag == 8)[0]  # EC_NEWTON_FAIL = 8
print(f"{'='*70}")
print(f"1A. TERMINAL PORTFOLIO FAILURES (age {pc.ages[-1]})")
print(f"{'='*70}")
print(f"Failed: {len(failed_terminal)} / {pc.N_state} states")
print()
if len(failed_terminal) > 0:
    print(f"  {'i_s':>4}  {'alpha_s':>7}  {'alpha_b':>7}  {'c_min':>8}  {'c_max':>8}  "
          + "  ".join(f"{n:>8}" for n in model.state_names))
    print(f"  {'-'*4}  {'-'*7}  {'-'*7}  {'-'*8}  {'-'*8}  "
          + "  ".join("-"*8 for _ in model.state_names))
    for i_s in failed_terminal:
        sv = pc.state_grid[i_s]
        a_s = float(S_mat[-1, 0, i_s, pc.n_w//2])  # terminal stock share
        a_b = float(B_mat[-1, 0, i_s, pc.n_w//2])  # terminal bond share
        c_min = float(C_mat[-1, 0, i_s, :].min())
        c_max = float(C_mat[-1, 0, i_s, :].max())
        sv_str = "  ".join(f"{sv[k]:+8.4f}" for k in range(len(model.state_names)))
        print(f"  {i_s:4d}  {a_s:7.3f}  {a_b:7.3f}  {c_min:8.2e}  {c_max:8.2e}  {sv_str}")

# Check overlap: do terminal failures match the age 79 failures?
failed_79 = set(np.where(fail_reasons[:, 3] > 0)[0])
failed_T = set(failed_terminal.tolist())
overlap = failed_79 & failed_T
print(f"\n  Terminal failures also failing at age 79: {len(overlap)} / {len(failed_T)}")
print(f"  Age 79 failures NOT in terminal set: {len(failed_79 - failed_T)}")

# -- 1B. Return magnitudes at failing vs non-failing states ------------------

print(f"\n{'='*70}")
print(f"1B. RETURN MAGNITUDES AT FAILING STATES")
print(f"{'='*70}")
print(f"\nGross portfolio returns R_stock = R_bill * exp(mu_r[i_s, j_s, 0]),")
print(f"                        R_bond  = R_bill * exp(mu_r[i_s, j_s, 1])")
print()

# Pick representative failing and non-failing states
worst_failing = [i for i in range(pc.N_state) if fail_reasons[i, 3] == 1331][:5]
moderate_failing = [i for i in range(pc.N_state)
                    if 0 < fail_reasons[i, 3] < 1000][:5]
non_failing = [i for i in range(pc.N_state) if fail_reasons[i, 3] == 0][:5]

for label, states in [("WORST FAILING (1331/1650)", worst_failing),
                       ("MODERATE FAILING", moderate_failing),
                       ("NON-FAILING", non_failing)]:
    print(f"  --- {label} ---")
    for i_s in states:
        R_bill = np.exp(pc.r_bill_grid[i_s])
        Rx_s = np.exp(pc.mu_r[i_s, :, 0])
        Rx_b = np.exp(pc.mu_r[i_s, :, 1])
        R_stock = R_bill * Rx_s
        R_bond = R_bill * Rx_b

        # Probability-weighted mean return
        pi = pc.Pi_state[i_s, :]
        E_Rs = np.sum(pi * R_stock)
        E_Rb = np.sum(pi * R_bond)

        sv = pc.state_grid[i_s]
        sv_str = "  ".join(f"{n}={sv[k]:+.4f}" for k, n in enumerate(model.state_names))
        print(f"  i_s={i_s:3d}  R_bill={R_bill:.4f}  "
              f"R_stock=[{R_stock.min():.3f}, {R_stock.max():.3f}] E={E_Rs:.3f}  "
              f"R_bond=[{R_bond.min():.3f}, {R_bond.max():.3f}] E={E_Rb:.3f}  "
              f"| {sv_str}")
    print()

# -- 1C. Extreme return transitions -----------------------------------------

print(f"{'='*70}")
print(f"1C. EXTREME RETURN TRANSITIONS (top/bottom 5 by R_stock and R_bond)")
print(f"{'='*70}")

# Collect all gross returns across all (i_s, j_s) pairs
all_R_stock = np.zeros((pc.N_state, pc.N_state))
all_R_bond = np.zeros((pc.N_state, pc.N_state))
for i_s in range(pc.N_state):
    R_bill = np.exp(pc.r_bill_grid[i_s])
    all_R_stock[i_s, :] = R_bill * np.exp(pc.mu_r[i_s, :, 0])
    all_R_bond[i_s, :] = R_bill * np.exp(pc.mu_r[i_s, :, 1])

print(f"\n  R_stock overall: [{all_R_stock.min():.4f}, {all_R_stock.max():.4f}]")
print(f"  R_bond  overall: [{all_R_bond.min():.4f}, {all_R_bond.max():.4f}]")

# Top 5 extreme stock returns
flat_idx = np.argsort(all_R_stock.ravel())
print(f"\n  Bottom 5 R_stock transitions:")
for k in range(5):
    idx = flat_idx[k]
    i_s, j_s = divmod(idx, pc.N_state)
    print(f"    i_s={i_s:3d} -> j_s={j_s:3d}:  R_stock={all_R_stock[i_s,j_s]:.4f}"
          f"  pi={pc.Pi_state[i_s,j_s]:.4e}"
          f"  | from dp={pc.state_grid[i_s,2]:.3f} to dp={pc.state_grid[j_s,2]:.3f}")

print(f"\n  Top 5 R_stock transitions:")
for k in range(5):
    idx = flat_idx[-(k+1)]
    i_s, j_s = divmod(idx, pc.N_state)
    print(f"    i_s={i_s:3d} -> j_s={j_s:3d}:  R_stock={all_R_stock[i_s,j_s]:.4f}"
          f"  pi={pc.Pi_state[i_s,j_s]:.4e}"
          f"  | from dp={pc.state_grid[i_s,2]:.3f} to dp={pc.state_grid[j_s,2]:.3f}")

print(f"\n  Bottom 5 R_bond transitions:")
flat_idx_b = np.argsort(all_R_bond.ravel())
for k in range(5):
    idx = flat_idx_b[k]
    i_s, j_s = divmod(idx, pc.N_state)
    print(f"    i_s={i_s:3d} -> j_s={j_s:3d}:  R_bond={all_R_bond[i_s,j_s]:.4f}"
          f"  pi={pc.Pi_state[i_s,j_s]:.4e}"
          f"  | from y_nom={pc.state_grid[i_s,1]:.4f} to y_nom={pc.state_grid[j_s,1]:.4f}")

print(f"\n  Top 5 R_bond transitions:")
for k in range(5):
    idx = flat_idx_b[-(k+1)]
    i_s, j_s = divmod(idx, pc.N_state)
    print(f"    i_s={i_s:3d} -> j_s={j_s:3d}:  R_bond={all_R_bond[i_s,j_s]:.4f}"
          f"  pi={pc.Pi_state[i_s,j_s]:.4e}"
          f"  | from y_nom={pc.state_grid[i_s,1]:.4f} to y_nom={pc.state_grid[j_s,1]:.4f}")

# -- 1D. Continuation consumption at failing states -------------------------

print(f"\n{'='*70}")
print(f"1D. CONTINUATION CONSUMPTION (c_next at age 80) FOR FAILING STATES")
print(f"{'='*70}")
print(f"\nAt failing i_s, what does the age-80 consumption policy look like?")
print(f"If c_next is tiny, c^(-gamma) blows up and FOCs become ~1e+9.\n")

z_med = pc.n_z // 2
w_med = pc.n_w // 2
# Check c_next at a range of wealth points
w_check = [0, pc.n_w//4, pc.n_w//2, 3*pc.n_w//4, pc.n_w-1]
w_vals = pc.wealth_grid[w_check]

print(f"  {'i_s':>4}  {'fail%':>5}  " +
      "  ".join(f"c(W={w:.2f})" for w in w_vals) +
      f"  {'state':>30}")
print(f"  {'-'*4}  {'-'*5}  " +
      "  ".join("-"*max(10, len(f"c(W={w:.2f})")) for w in w_vals) +
      f"  {'-'*30}")

# Show all failing states + a few non-failing for comparison
states_to_show = sorted(set(
    list(np.where(fail_reasons[:, 3] > 1000)[0][:10]) +  # worst failing
    list(np.where((fail_reasons[:, 3] > 0) & (fail_reasons[:, 3] < 1000))[0][:5]) +  # moderate
    list(np.where(fail_reasons[:, 3] == 0)[0][:3])  # non-failing for reference
))

for i_s in states_to_show:
    nf = int(fail_reasons[i_s, 3])
    nc = int(fail_reasons[i_s, 4])
    pct = f"{100*nf/nc:.0f}%" if nc > 0 else "N/A"
    c_vals = C_mat[-1, z_med, i_s, w_check]
    c_str = "  ".join(f"{c:10.2e}" for c in c_vals)
    sv = pc.state_grid[i_s]
    sv_short = " ".join(f"{n}={sv[k]:+.3f}" for k, n in enumerate(model.state_names))
    marker = " <-- OK" if nf == 0 else ""
    print(f"  {i_s:4d}  {pct:>5}  {c_str}  {sv_short}{marker}")

# -- 1E. Sign pattern of excess returns at worst states ---------------------

print(f"\n{'='*70}")
print(f"1E. EXCESS RETURN SIGN PATTERN AT WORST STATES")
print(f"{'='*70}")
print(f"\nFor states where ALL corner FOCs are positive (no bracket),")
print(f"check how many transitions have positive excess returns.\n")

for i_s in worst_failing[:5]:
    R_bill = np.exp(pc.r_bill_grid[i_s])
    Rx_s = np.exp(pc.mu_r[i_s, :, 0])
    Rx_b = np.exp(pc.mu_r[i_s, :, 1])
    Rex_s = R_bill * Rx_s - R_bill  # = R_bill * (Rx_s - 1)
    Rex_b = R_bill * Rx_b - R_bill

    pi = pc.Pi_state[i_s, :]
    active = pi > 1e-10
    n_active = active.sum()
    n_pos_s = (Rex_s[active] > 0).sum()
    n_pos_b = (Rex_b[active] > 0).sum()
    n_both_pos = ((Rex_s[active] > 0) & (Rex_b[active] > 0)).sum()

    sv = pc.state_grid[i_s]
    print(f"  i_s={i_s:3d} (dp={sv[2]:+.3f}):  "
          f"Rex_s>0: {n_pos_s}/{n_active}  "
          f"Rex_b>0: {n_pos_b}/{n_active}  "
          f"Both>0: {n_both_pos}/{n_active}  "
          f"Rex_s range=[{Rex_s[active].min():+.3f},{Rex_s[active].max():+.3f}]  "
          f"Rex_b range=[{Rex_b[active].min():+.3f},{Rex_b[active].max():+.3f}]")

# -- 1F. Summary ------------------------------------------------------------

print(f"\n{'='*70}")
print(f"1F. SUMMARY")
print(f"{'='*70}")
n_all_pos = 0
for i_s in range(pc.N_state):
    if fail_reasons[i_s, 3] > 1000:
        R_bill = np.exp(pc.r_bill_grid[i_s])
        Rex_s = R_bill * (np.exp(pc.mu_r[i_s, :, 0]) - 1)
        Rex_b = R_bill * (np.exp(pc.mu_r[i_s, :, 1]) - 1)
        pi = pc.Pi_state[i_s, :]
        active = pi > 1e-10
        if (Rex_s[active] > 0).all() and (Rex_b[active] > 0).all():
            n_all_pos += 1

n_worst = int((fail_reasons[:, 3] > 1000).sum())
print(f"  Worst-failing states (>1000 failures): {n_worst}")
print(f"  Of these, states where ALL active transitions have Rex_s>0 AND Rex_b>0: {n_all_pos}")
print(f"  Terminal failures: {len(failed_terminal)}")
print(f"  Age 79 failures at non-terminal-failure states: {len(failed_79 - failed_T)}")
print(f"\n  If most worst-failing states have universally positive excess returns,")
print(f"  the issue is that K=1 (point-mass returns) creates NO downside risk")
print(f"  at extreme states, so the agent always wants >100% risky allocation.")
print(f"  Fix: add a 'best feasible corner' fallback when no KKT point exists.")
print(f"{'='*70}")

1A. TERMINAL PORTFOLIO FAILURES (age 80)
Failed: 0 / 125 states


  Terminal failures also failing at age 79: 0 / 0
  Age 79 failures NOT in terminal set: 33

1B. RETURN MAGNITUDES AT FAILING STATES

Gross portfolio returns R_stock = R_bill * exp(mu_r[i_s, j_s, 0]),
                        R_bond  = R_bill * exp(mu_r[i_s, j_s, 1])

  --- WORST FAILING (1331/1650) ---

  --- MODERATE FAILING ---
  i_s=  1  R_bill=0.9859  R_stock=[0.531, 1.341] E=1.027  R_bond=[0.407, 1.008] E=0.937  | rtb=-0.0142  y_nom=-0.0019  dp=-4.3856
  i_s=  6  R_bill=0.9859  R_stock=[0.519, 1.310] E=0.994  R_bond=[0.506, 1.254] E=0.966  | rtb=-0.0142  y_nom=+0.0036  dp=-4.3856
  i_s= 10  R_bill=0.9859  R_stock=[0.405, 1.022] E=0.911  R_bond=[0.629, 1.557] E=0.994  | rtb=-0.0142  y_nom=+0.0091  dp=-4.6227
  i_s= 11  R_bill=0.9859  R_stock=[0.507, 1.279] E=0.962  R_bond=[0.630, 1.560] E=0.997  | rtb=-0.0142  y_nom=+0.0091  dp=-4.3856
  i_s= 12  R_bill=0.9859  R_stock=[0.635, 1.602] E=1.016  R_bond=[0.631, 1.563] E=